# APS360 Handwriting Recognition: Step 2 (Train the CRNN)

Trains the CRNN + CTC recognizer from the proposal on IAM words, then evaluates it.

| Section | What happens |
|---|---|
| Setup | GPU check, Drive, code, data (reuses `words_recovered.txt` from Step 1) |
| Model code | the CRNN, training loop, evaluation, sweep (readable cells) |
| Sanity check | the model memorises one batch: proves the model, loss and decoding are wired up correctly |
| Sweep | 12 short runs (lr × hidden grid + dropout / batch / hidden / depth variations), ranked by **validation** CER; resumable |
| Train | uses the sweep winner (or the defaults), full run with checkpoints on Drive; re-run to resume |
| Evaluate | CER / WER on val and **test**, speed per word, example predictions, common mistakes |
| Report outputs | every figure, table and number the progress report needs, saved to `outputs/report/` on Drive |

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
**Run Step 1 first** (at least once) so `words_recovered.txt` exists; otherwise this trains on the cut-off subset only.

## Check the GPU
You should see a Tesla T4. If it says CPU, change the runtime type: training on CPU would take many hours.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run all.")

## Mount Google Drive
Your Drive appears at `/content/drive/MyDrive`. Allow access when Colab asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paths
`PROJECT_DIR` is the Drive folder that holds `archive.zip`. If it has been moved, the cell searches your Drive for it.
Heavy files go on Colab's local disk (`/content`), which is far faster than Drive. Results are saved back to Drive.

In [ ]:
import os, subprocess
PROJECT_DIR = "/content/drive/MyDrive/University (Mech Eng)/Year 3/Fall/APS360 (Introduction to Machine Learning)/Project/Project Progress Report"
if not os.path.exists(os.path.join(PROJECT_DIR, "archive.zip")):
    print("archive.zip not found at PROJECT_DIR; searching Drive (can take a minute)...")
    hits = [h for h in subprocess.run(["find", "/content/drive/MyDrive", "-name", "archive.zip", "-size", "+500M"],
                                      capture_output=True, text=True).stdout.split("\n") if h]
    assert hits, "Could not find archive.zip anywhere in your Drive"
    PROJECT_DIR = os.path.dirname(hits[0])

ARCHIVE    = os.path.join(PROJECT_DIR, "archive.zip")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")       # results saved back to Drive
REPORT_DIR = os.path.join(OUTPUT_DIR, "report")         # figures/tables for the progress report
OFFICIAL_WORDS_TXT = os.path.join(PROJECT_DIR, "words.txt")            # full official file, if you add it
RECOVERED_WORDS_TXT = os.path.join(PROJECT_DIR, "words_recovered.txt") # made by the label-recovery step
CODE_DIR   = "/content/aps360_htr"                      # project code (this session only)
RAW_DIR    = "/content/iam_raw"                         # archive.zip unpacked here
DATA_ROOT  = "/content/iam"                             # tidy IAM layout (symlinks into RAW_DIR)
os.environ["HTR_DATA_ROOT"] = DATA_ROOT                 # src/config.py reads this
os.makedirs(REPORT_DIR, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)

## Write the project code
The model and training code is shown below as `%%writefile` cells. The data code from Step 1 (identical) comes from the embedded zip.

| File | What it does |
|---|---|
| `src/models/crnn.py` | the CRNN; `output_lengths()` gives CTC the real number of time steps per image |
| `src/training/sampler.py` | puts words of similar width in the same batch (less padding, faster) |
| `src/training/engine.py` | `fit()` (AdamW, gradient clipping, mixed precision, LR halving, early stopping, resume), `evaluate()`, checkpoints, `overfit_one_batch()` |
| `src/training/sweep.py` | the 12-run sweep (resumable), winner selection with fallback to defaults, run-folder check |
| `src/training/data.py` | `load_data()`: Step 1's `make_datasets()` + preloading images into RAM |
| `src/utils/plots.py` | training curves and prediction grids |
| `src/report/*.py` | report figures (PNG + PDF), LaTeX tables, error analysis, model summary, `report_numbers.json` |
| `scripts/train.py`, `evaluate.py`, `sweep.py` | command-line versions of the same steps |

### The rest of the codebase (embedded zip)
Every step notebook is generated from one repo; this one embeds code **v16** (`aps360_htr_code_v16.zip` in your project folder).
This cell unpacks the 61 files that aren't shown below (see the other notebook or the zip to read them).
The long string is just that zip, base64-encoded.

In [ ]:
import base64, io, os, shutil, sys, zipfile
# Start from a clean copy every time: delete any older code left in /content/aps360_htr,
# and forget already-imported project modules so this session can't keep using stale code.
os.chdir("/content")
shutil.rmtree("/content/aps360_htr", ignore_errors=True)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
os.makedirs("/content/aps360_htr", exist_ok=True)
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall("/content/aps360_htr")
with open("/content/aps360_htr/CODE_VERSION", "w") as f:
    f.write("v16\n")
print("unpacked", len(zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).namelist()), "files | code v16")

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/models/__init__.py', 'src/models/crnn.py', 'src/config.py', 'src/training/__init__.py', 'src/training/sampler.py', 'src/training/engine.py', 'src/training/data.py', 'src/training/sweep.py', 'src/utils/plots.py', 'src/report/__init__.py', 'src/report/style.py', 'src/report/tables.py', 'src/report/errors.py', 'src/report/figures.py', 'src/report/model_summary.py', 'src/report/predict.py', 'src/report/manifest.py', 'src/report/crnn_report.py', 'src/report/compare.py', 'scripts/train.py', 'scripts/evaluate.py', 'scripts/sweep.py']}):
    os.makedirs(f'/content/aps360_htr/{d}', exist_ok=True)
%cd /content/aps360_htr

### The CRNN model
CNN (5 conv → BatchNorm → ReLU blocks) → 2-layer bidirectional LSTM → linear layer, with log-softmax over the 78 characters + blank. Width shrinks by 4, so a 128-px-wide word gives 32 time steps.

In [ ]:
%%writefile /content/aps360_htr/src/models/__init__.py
"""Models (CRNN comes in step 3)."""

In [ ]:
%%writefile /content/aps360_htr/src/models/crnn.py
"""CRNN: CNN feature extractor -> 2-layer BiLSTM -> linear layer, trained with CTC.

Shapes for one batch (B images, padded width W):

    image          (B, 1, 32, W)
    after the CNN  (B, 512, 1, W/4)   height squeezed to 1, width / 4
    as a sequence  (T=W/4, B, 512)    one feature vector per 4-pixel-wide column
    after BiLSTM   (T, B, 2*hidden)   each step sees the whole word (both directions)
    output         (T, B, C)          log-probabilities over C = charset + blank

The output is exactly what nn.CTCLoss wants, and Charset.decode_batch decodes it.
"""
import torch
import torch.nn as nn

from src.config import ModelConfig


def conv_block(c_in, c_out, pool):
    """3x3 conv -> BatchNorm -> ReLU, then optional max-pool.

    pool=(2, 2) halves height and width; pool=(2, 1) halves only the height,
    which keeps more time steps for CTC (long words need many steps).
    """
    layers = [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
              nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]
    if pool:
        layers.append(nn.MaxPool2d(pool, pool))
    return nn.Sequential(*layers)


class CRNN(nn.Module):
    # Width shrinks by 2 in the first two blocks only, so T = W // 4.
    WIDTH_DOWNSAMPLE = 4

    def __init__(self, num_classes: int, cfg: ModelConfig | None = None):
        super().__init__()
        cfg = cfg or ModelConfig()
        c = cfg.cnn_channels
        # Height: 32 -> 16 -> 8 -> 4 -> 2 -> 1.  Width: W -> W/2 -> W/4 (then unchanged).
        self.cnn = nn.Sequential(
            conv_block(1,    c[0], (2, 2)),
            conv_block(c[0], c[1], (2, 2)),
            conv_block(c[1], c[2], (2, 1)),
            conv_block(c[2], c[3], (2, 1)),
            conv_block(c[3], c[4], (2, 1)),
        )
        self.dropout = nn.Dropout(cfg.dropout)
        self.rnn = nn.LSTM(c[4], cfg.lstm_hidden, num_layers=cfg.lstm_layers,
                           bidirectional=True, dropout=cfg.dropout if cfg.lstm_layers > 1 else 0.0)
        self.fc = nn.Linear(2 * cfg.lstm_hidden, num_classes)

    def forward(self, x):
        f = self.cnn(x)                  # (B, C, 1, T)
        f = f.squeeze(2).permute(2, 0, 1)  # (T, B, C): time first, as the LSTM and CTC expect
        f, _ = self.rnn(self.dropout(f))
        logits = self.fc(self.dropout(f))  # (T, B, num_classes)
        return logits.log_softmax(dim=2)

    @classmethod
    def output_lengths(cls, widths: torch.Tensor) -> torch.Tensor:
        """Real (unpadded) number of time steps for each image: its width // 4.
        CTC uses these so it ignores the columns that are only padding."""
        return torch.clamp(widths // cls.WIDTH_DOWNSAMPLE, min=1)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Settings (model + training defaults)

In [ ]:
%%writefile /content/aps360_htr/src/config.py
"""Central settings for the project.

Every script reads its defaults from here, so a change in one place applies
everywhere. Command-line flags in the scripts can override most of them.
"""
import os
from dataclasses import dataclass, field
from pathlib import Path

# Repo root = the folder that contains src/ (works on a laptop or in Colab).
ROOT = Path(__file__).resolve().parents[1]

# Where the IAM data lives. Set the HTR_DATA_ROOT environment variable (or pass
# --data-root to the scripts) to point somewhere else, e.g. in Colab:
#   os.environ["HTR_DATA_ROOT"] = "/content/iam"
#   or "/content/drive/MyDrive/APS360/iam" (works, but reading from Drive is slow)
DATA_ROOT = Path(os.environ.get("HTR_DATA_ROOT", ROOT / "data" / "iam"))


@dataclass
class DataConfig:
    # Where IAM lives. Expected layout (same for the fake stand-in data):
    #   <iam_root>/ascii/words.txt
    #   <iam_root>/ascii/forms.txt            (optional, gives writer IDs)
    #   <iam_root>/words/a01/a01-000u/a01-000u-00-00.png
    iam_root: Path = DATA_ROOT

    # Optional split lists (one ID per line: form, line or word ID).
    # Defaults to the Aachen splits (form IDs) that ship in data/splits/aachen.
    # Set split_dir=None to always use the automatic writer-disjoint split.
    split_dir: Path | None = ROOT / "data" / "splits" / "aachen"
    split_files: dict = field(default_factory=lambda: {
        "train": "train.uttlist",
        "val": "validation.uttlist",
        "test": "test.uttlist",
    })

    # Form -> writer table, used when the download has no ascii/forms.txt (e.g. Kaggle).
    # See data/iam_form_writers_SOURCE.md for where it comes from. None = don't use it.
    form_writers_csv: Path | None = ROOT / "data" / "iam_form_writers.csv"

    # Fallback split (used when split files are missing): fractions of items
    # per split, grouped so that no writer appears in two splits.
    split_fractions: tuple = (0.8, 0.1, 0.1)   # train, val, test
    split_seed: int = 42

    # Filtering
    drop_err: bool = True        # drop words whose segmentation status is "err"
    max_label_len: int = 32      # drop longer words (also keeps CTC happy)
    check_images: bool = True    # open every image once to drop broken/empty files

    # Image preprocessing
    img_height: int = 32         # brief: resize to 32 px high
    min_width: int = 16          # very short words (e.g. ".") get at least this
    max_width: int = 512         # very long words get squeezed to this

    # Loader
    batch_size: int = 32
    num_workers: int = 0         # 0 is safest on Windows/macOS; raise on Linux


@dataclass
class ModelConfig:
    """CRNN size. Saved inside every checkpoint so evaluate.py can rebuild it."""
    cnn_channels: tuple = (64, 128, 256, 256, 512)  # the 5 conv blocks
    lstm_hidden: int = 256       # per direction (BiLSTM output = 2x this)
    lstm_layers: int = 2
    dropout: float = 0.3


@dataclass
class TrainConfig:
    epochs: int = 40
    batch_size: int = 64
    lr: float = 1e-3             # peak learning rate (AdamW)
    weight_decay: float = 1e-4
    grad_clip: float = 5.0       # LSTMs + CTC can produce the odd huge gradient
    patience: int = 8            # stop if val CER hasn't improved for this many epochs
    lr_patience: int = 3         # halve the LR after this many epochs without improvement
    amp: bool = True             # mixed precision on GPU (faster on a T4)
    num_workers: int = 2         # Colab has 2 CPU cores
    preload: bool = True         # keep resized images in RAM (much faster epochs)
    train_frac: float = 1.0      # <1 trains on a random subset (quick sweeps)
    max_batches: int | None = None  # cap batches per epoch (smoke tests)
    seed: int = 42
    out_dir: Path = ROOT / "outputs" / "runs" / "crnn"

### Batching words of similar width

In [ ]:
%%writefile /content/aps360_htr/src/training/__init__.py
"""Training loop, evaluation, batching and checkpoints."""

In [ ]:
%%writefile /content/aps360_htr/src/training/sampler.py
"""Batch words of similar width together.

A batch is padded to its widest image, so mixing "a" with "understanding" wastes
most of the compute on padding. We shuffle, cut the data into big chunks, sort
each chunk by width, then slice it into batches and shuffle the batches. Every
epoch is still random, but there's much less padding (often 2x faster).
"""
import random

from torch.utils.data import Sampler


class BucketBatchSampler(Sampler):
    def __init__(self, widths, batch_size, shuffle=True, chunk_batches=50, seed=0, drop_last=False):
        self.widths, self.batch_size, self.shuffle = list(widths), batch_size, shuffle
        self.chunk = batch_size * chunk_batches
        self.seed, self.epoch, self.drop_last = seed, 0, drop_last

    def set_epoch(self, epoch: int):
        self.epoch = epoch   # different shuffle each epoch, same across runs

    def _batches(self):
        idx = list(range(len(self.widths)))
        rng = random.Random(self.seed + self.epoch)
        if self.shuffle:
            rng.shuffle(idx)
        batches = []
        for start in range(0, len(idx), self.chunk):
            chunk = sorted(idx[start:start + self.chunk], key=lambda i: self.widths[i])
            for b in range(0, len(chunk), self.batch_size):
                batch = chunk[b:b + self.batch_size]
                if len(batch) == self.batch_size or not self.drop_last:
                    batches.append(batch)
        if self.shuffle:
            rng.shuffle(batches)
        return batches

    def __iter__(self):
        return iter(self._batches())

    def __len__(self):
        return len(self._batches())

### Training loop, evaluation, checkpoints

In [ ]:
%%writefile /content/aps360_htr/src/training/engine.py
"""Training and evaluation for the CRNN.

The pieces, in the order a training run uses them:
  make_loader       batches of similar width (BucketBatchSampler) + collate_fn
  ctc_loss_on_batch forward pass + CTC loss for one batch
  train_one_epoch   AdamW step per batch, gradient clipping, mixed precision on GPU
  evaluate          greedy decoding -> CER / WER (+ every prediction)
  fit               the epoch loop: LR schedule, best checkpoint, early stopping, resume
"""
import csv
import json
import math
import random
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.config import ModelConfig, TrainConfig
from src.data.charset import Charset
from src.data.iam import collate_fn
from src.models.crnn import CRNN, count_parameters
from src.training.sampler import BucketBatchSampler
from src.utils.metrics import cer, wer


def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------------ data

def subset(ds, frac: float, seed: int = 0):
    """Random fraction of a dataset (for quick sweeps). Keeps .widths() working."""
    if frac >= 1.0:
        return ds
    idx = sorted(random.Random(seed).sample(range(len(ds)), max(1, int(len(ds) * frac))))
    sub = Subset(ds, idx)
    sub.widths = lambda: _pick(ds.widths(), idx)
    return sub


def _pick(values, idx):
    return [values[i] for i in idx]


def make_loader(ds, batch_size, shuffle, num_workers=0, seed=0):
    widths = ds.widths()
    sampler = BucketBatchSampler(widths, batch_size, shuffle=shuffle, seed=seed)
    loader = DataLoader(ds, batch_sampler=sampler, collate_fn=collate_fn, num_workers=num_workers,
                        pin_memory=torch.cuda.is_available(), persistent_workers=num_workers > 0)
    return loader, sampler


# ------------------------------------------------------------------ one batch

ctc_loss_fn = nn.CTCLoss(blank=0, reduction="mean", zero_infinity=True)
# zero_infinity: if a label can't fit in the time steps (word too long for its
# width), that sample gives loss 0 instead of inf, so one bad sample can't wreck training.


def ctc_loss_on_batch(model, batch, device, amp=False):
    images = batch["images"].to(device, non_blocking=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
        log_probs = model(images)                         # (T, B, C)
    input_lengths = CRNN.output_lengths(batch["widths"])  # real time steps per image
    # CTC is computed in float32: it's numerically touchy in half precision.
    loss = ctc_loss_fn(log_probs.float(), batch["targets"].to(device), input_lengths, batch["target_lengths"])
    return loss, log_probs, input_lengths


# ------------------------------------------------------------------ epochs

def train_one_epoch(model, loader, optimizer, scaler, device, cfg: TrainConfig, log_every=100):
    model.train()
    total, n, t0 = 0.0, 0, time.time()
    for i, batch in enumerate(loader):
        if cfg.max_batches and i >= cfg.max_batches:
            break
        if batch is None:                               # every image in it was unreadable
            continue
        loss, _, _ = ctc_loss_on_batch(model, batch, device, amp=scaler.is_enabled())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                      # so clipping sees real gradient sizes
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(batch["texts"]); n += len(batch["texts"])
        if log_every and (i + 1) % log_every == 0:
            print(f"    batch {i+1}/{len(loader)}  loss {total/n:.3f}  ({time.time()-t0:.0f}s)")
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, charset: Charset, device, amp=False, max_batches=None):
    """Greedy-decode a whole split. Returns dict with loss, cer, wer, predictions."""
    model.eval()
    refs, hyps, total, n = [], [], 0.0, 0
    t0 = time.time()
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        if batch is None:                                # every image in it was unreadable
            continue
        loss, log_probs, lengths = ctc_loss_on_batch(model, batch, device, amp=amp)
        best = log_probs.argmax(dim=2).cpu()             # (T, B)
        for b, L in enumerate(lengths.tolist()):
            hyps.append(charset.ctc_greedy_decode(best[:L, b].tolist()))
        refs += batch["texts"]
        total += loss.item() * len(batch["texts"]); n += len(batch["texts"])
    secs = time.time() - t0
    return {"loss": total / max(n, 1), "cer": cer(refs, hyps), "wer": wer(refs, hyps),
            "n": len(refs), "ms_per_word": 1000 * secs / max(len(refs), 1),
            "refs": refs, "hyps": hyps}


# ------------------------------------------------------------------ checkpoints

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_cer, history,
                    model_cfg: ModelConfig, charset: Charset, bad_epochs=0):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "epoch": epoch, "best_cer": best_cer, "history": history, "bad_epochs": bad_epochs,
                "model_cfg": asdict(model_cfg), "charset": charset.chars}, tmp)
    tmp.replace(path)   # write then rename: a Colab disconnect can't leave a half-written file


def load_model(path, device=None):
    """Rebuild the CRNN from a checkpoint. Returns (model, charset, checkpoint dict)."""
    device = device or get_device()
    ck = torch.load(path, map_location=device, weights_only=False)
    mc = ck["model_cfg"]; mc["cnn_channels"] = tuple(mc["cnn_channels"])
    charset = Charset(ck["charset"])
    model = CRNN(charset.num_classes, ModelConfig(**mc)).to(device)
    model.load_state_dict(ck["model"])
    model.eval()
    return model, charset, ck


# ------------------------------------------------------------------ the full run

def fit(datasets, charset: Charset, model_cfg: ModelConfig, cfg: TrainConfig, resume=True, verbose=True):
    """Train on datasets['train'], pick the best epoch by val CER.

    Saves to cfg.out_dir:  best.pt (lowest val CER), last.pt (for resuming),
    history.csv (one row per epoch) and config.json.
    """
    seed_everything(cfg.seed)
    device = get_device()
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    amp = cfg.amp and device.type == "cuda"

    train_ds = subset(datasets["train"], cfg.train_frac, cfg.seed)
    train_loader, train_sampler = make_loader(train_ds, cfg.batch_size, True, cfg.num_workers, cfg.seed)
    val_loader, _ = make_loader(datasets["val"], cfg.batch_size * 2, False, cfg.num_workers)

    model = CRNN(charset.num_classes, model_cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    # Halve the LR when val CER stops improving for lr_patience epochs.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
                                                           patience=cfg.lr_patience)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    start, best_cer, history, bad = 1, math.inf, [], 0

    last = out / "last.pt"
    if resume and last.exists():
        ck = torch.load(last, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
        start, best_cer, history = ck["epoch"] + 1, ck["best_cer"], ck["history"]
        bad = ck.get("bad_epochs", 0)
        print(f"Resuming from epoch {ck['epoch']} (best val CER so far {best_cer:.4f})")

    (out / "config.json").write_text(json.dumps(
        {"model": asdict(model_cfg), "train": {k: str(v) for k, v in asdict(cfg).items()},
         "train_words": len(train_ds), "val_words": len(datasets["val"]),
         "parameters": count_parameters(model), "device": str(device)}, indent=2))
    if verbose:
        print(f"device {device} | {count_parameters(model):,} parameters | "
              f"{len(train_ds)} train / {len(datasets['val'])} val words | {len(train_loader)} batches/epoch")

    for epoch in range(start, cfg.epochs + 1):
        if bad >= cfg.patience:
            print(f"Early stop: val CER hasn't improved for {cfg.patience} epochs"); break
        t0 = time.time()
        train_sampler.set_epoch(epoch)
        if hasattr(train_ds, "set_epoch"):          # Model B: new random augmentations each epoch
            train_ds.set_epoch(epoch)
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device, cfg,
                                     log_every=100 if verbose else 0)
        val = evaluate(model, val_loader, charset, device, amp=amp, max_batches=cfg.max_batches)
        scheduler.step(val["cer"])
        improved = val["cer"] < best_cer
        if improved:
            best_cer, bad = val["cer"], 0
        else:
            bad += 1
        row = {"epoch": epoch, "train_loss": round(train_loss, 4), "val_loss": round(val["loss"], 4),
               "val_cer": round(val["cer"], 4), "val_wer": round(val["wer"], 4),
               "lr": optimizer.param_groups[0]["lr"], "secs": round(time.time() - t0, 1)}
        history.append(row)
        if improved:
            save_checkpoint(out / "best.pt", model, optimizer, scheduler, scaler, epoch, best_cer,
                            history, model_cfg, charset, bad)
        save_checkpoint(last, model, optimizer, scheduler, scaler, epoch, best_cer, history,
                        model_cfg, charset, bad)
        with open(out / "history.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerows(history)
        if verbose:
            k = max(1, len(val["refs"]) // 4)          # 3 examples spread across the val set
            ex = ", ".join(f"{r!r}->{h!r}" for r, h in list(zip(val["refs"], val["hyps"]))[k::k][:3])
            print(f"epoch {epoch:3d} | train loss {train_loss:.3f} | val loss {val['loss']:.3f} | "
                  f"val CER {val['cer']:.4f} WER {val['wer']:.4f} | lr {row['lr']:.1e} | "
                  f"{row['secs']:.0f}s {'*best*' if improved else ''}\n          e.g. {ex}")
    return {"best_val_cer": best_cer, "history": history, "out_dir": str(out)}


def overfit_one_batch(dataset, charset: Charset, model_cfg=None, steps=200, batch_size=16, lr=1e-3):
    """Sanity check: a correct model + loss should memorise one small batch.
    Returns the list of losses (should fall towards 0) and the final predictions."""
    seed_everything(0)
    device = get_device()
    batch = collate_fn([dataset[i] for i in range(min(batch_size, len(dataset)))])
    model = CRNN(charset.num_classes, model_cfg or ModelConfig(dropout=0.0)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    model.train()
    for _ in range(steps):
        loss, _, _ = ctc_loss_on_batch(model, batch, device)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        _, log_probs, lengths = ctc_loss_on_batch(model, batch, device)
    best = log_probs.argmax(2).cpu()
    preds = [charset.ctc_greedy_decode(best[:L, b].tolist()) for b, L in enumerate(lengths.tolist())]
    return losses, list(zip(batch["texts"], preds))

In [ ]:
%%writefile /content/aps360_htr/src/training/data.py
"""Build the datasets for training/evaluation (shared by the scripts and notebook)."""
import time
from pathlib import Path

from src.config import DataConfig
from src.data.charset import Charset
from src.data.iam import make_datasets


def add_dropped(stats, datasets):
    """Count images dropped while loading (unreadable files) as bad images, so the
    data-filtering table stays correct. Safe to call more than once."""
    seen = set(stats.get("bad_image_ids", []))
    new = [w for ds in datasets.values() for w in getattr(ds, "dropped", []) if w not in seen]
    if new:
        stats["bad_or_missing_image"] = stats.get("bad_or_missing_image", 0) + len(new)
        stats["kept"] = stats.get("kept", 0) - len(new)
        stats["bad_image_ids"] = sorted(seen | set(new))
        print(f"counted {len(new)} unreadable image(s) as bad images: {', '.join(new[:10])}")
    return stats


def load_data(data_root=None, splits=("train", "val", "test"), preload=True, check_images=True,
              split_dir="default"):
    """make_datasets() from step 1 (same filters, Aachen splits, writer table),
    then optionally preload the images of the splits we need into RAM."""
    cfg = DataConfig()
    if data_root:
        cfg.iam_root = Path(data_root)
    if split_dir != "default":
        cfg.split_dir = None if split_dir in (None, "none") else Path(split_dir)
    cfg.check_images = check_images
    charset = Charset()
    datasets, stats = make_datasets(cfg, charset)
    sizes = {s: len(datasets[s]) for s in datasets}
    print(f"words per split: {sizes} | split: {stats.get('split_method')}")
    if preload:
        for s in splits:
            t = time.time()
            datasets[s].preload()        # drops (and lists) any image that can't be decoded
            print(f"  preloaded {s}: {len(datasets[s])} images in {time.time() - t:.0f}s")
    add_dropped(stats, datasets)
    return datasets, charset, stats

### Hyperparameter sweep and picking the winner
`run_sweep()` (resumable), `choose_settings()` (winner or defaults) and `check_run_dir()` (never resume a checkpoint trained with other settings).

In [ ]:
%%writefile /content/aps360_htr/src/training/sweep.py
"""Hyperparameter sweep: short runs ranked by validation CER, and picking the winner.

Design (12 runs, not a full grid):
  * a small grid: learning rate {5e-4, 1e-3, 3e-3} x LSTM hidden {128, 256}   (6 runs)
  * one-at-a-time changes around the default (lr 1e-3, hidden 256, dropout 0.3,
    batch 64, 2 LSTM layers): dropout 0.1 / 0.5, batch 32 / 128, hidden 384,
    3 LSTM layers                                                              (6 runs)
Every run trains for the same number of epochs on the same 30% of the training words
(fixed seed), and is scored on the full validation set. The test set is never used.

Resumable: each finished run is appended to sweep_results.csv straight away, and runs
already in the CSV (same settings, epochs and train fraction) are skipped, so a Colab
disconnect only loses the run in progress (and that one resumes from its last.pt).
"""
import csv
import json
import math
import os
import shutil
import time
from pathlib import Path

from src.config import ModelConfig, TrainConfig
from src.models.crnn import CRNN, count_parameters

DEFAULTS = {"lr": 1e-3, "lstm_hidden": 256, "dropout": 0.3, "batch_size": 64, "lstm_layers": 2}
SWEPT = ("lr", "lstm_hidden", "dropout", "batch_size", "lstm_layers")
FIELDS = ["name", "group", *SWEPT, "epochs", "train_frac", "best_val_cer", "best_val_wer", "best_epoch",
          "epochs_run", "minutes", "parameters"]
_TYPES = {"lr": float, "dropout": float, "train_frac": float, "best_val_cer": float, "best_val_wer": float,
          "minutes": float, "lstm_hidden": int, "batch_size": int, "lstm_layers": int, "epochs": int,
          "best_epoch": int, "epochs_run": int, "parameters": int}


def run_name(s: dict) -> str:
    """Short unique name for a setting, e.g. lr0.001_h256_d0.3_b64_L2 (also used in RUN_NAME)."""
    return f"lr{s['lr']:g}_h{s['lstm_hidden']}_d{s['dropout']:g}_b{s['batch_size']}_L{s['lstm_layers']}"


def sweep_configs(defaults=DEFAULTS):
    """The 12 settings described at the top of this file, each with a 'group' label."""
    out = []
    for lr in (5e-4, 1e-3, 3e-3):
        for hidden in (128, 256):
            out.append({**defaults, "lr": lr, "lstm_hidden": hidden, "group": "grid: lr x hidden"})
    for key, values in (("dropout", (0.1, 0.5)), ("batch_size", (32, 128)),
                        ("lstm_hidden", (384,)), ("lstm_layers", (3,))):
        for v in values:
            out.append({**defaults, key: v, "group": f"vary {key}"})
    seen, unique = set(), []
    for c in out:
        if run_name(c) not in seen:
            seen.add(run_name(c)); unique.append(c)
    return unique


def load_results(csv_path):
    """Rows of a sweep_results.csv, with numbers parsed. Rows in an unknown format are dropped."""
    if not os.path.exists(csv_path):
        return []
    rows = []
    with open(csv_path, newline="") as f:
        for r in csv.DictReader(f):
            try:
                rows.append({k: _TYPES.get(k, str)(r[k]) for k in FIELDS})
            except (KeyError, ValueError, TypeError):
                continue
    return rows


def _legacy_aside(csv_path):
    """A sweep_results.csv from the old 4-run sweep has other columns: keep it, renamed."""
    if os.path.exists(csv_path):
        with open(csv_path, newline="") as f:
            header = next(csv.reader(f), [])
        if header and header != FIELDS:
            dst = str(csv_path).replace(".csv", "_old_format.csv")
            shutil.move(csv_path, dst)
            print(f"moved an older-format sweep file to {os.path.basename(dst)} (kept, not used)")


def _done(rows, cfg, epochs, train_frac):
    return any(r["name"] == run_name(cfg) and r["epochs"] == epochs and math.isclose(r["train_frac"], train_frac)
               for r in rows)


def run_sweep(datasets, charset, csv_path, out_root, configs=None, epochs=8, train_frac=0.3,
              max_batches=None, fit_fn=None, verbose=True):
    """Train every config that isn't in `csv_path` yet; append each result as soon as it finishes."""
    from src.training.engine import fit
    fit_fn = fit_fn or fit
    configs = configs or sweep_configs()
    _legacy_aside(csv_path)
    rows = load_results(csv_path)
    todo = [c for c in configs if not _done(rows, c, epochs, train_frac)]
    print(f"sweep: {len(configs)} settings, {len(configs) - len(todo)} already in {os.path.basename(str(csv_path))}, "
          f"{len(todo)} to run ({epochs} epochs on {train_frac:.0%} of train each)")
    t_start = time.time()
    for i, c in enumerate(todo, 1):
        name = run_name(c)
        mc = ModelConfig(lstm_hidden=c["lstm_hidden"], lstm_layers=c["lstm_layers"], dropout=c["dropout"])
        tc = TrainConfig(epochs=epochs, batch_size=c["batch_size"], lr=c["lr"], train_frac=train_frac,
                         patience=epochs, num_workers=0, max_batches=max_batches,
                         out_dir=Path(out_root) / f"{name}_e{epochs}_f{train_frac:g}")
        t0 = time.time()
        r = fit_fn(datasets, charset, mc, tc, resume=True, verbose=False)
        hist = r["history"]
        best = min(hist, key=lambda h: h["val_cer"]) if hist else {"val_cer": math.inf, "val_wer": math.inf, "epoch": 0}
        row = {"name": name, "group": c.get("group", ""), **{k: c[k] for k in SWEPT}, "epochs": epochs,
               "train_frac": train_frac, "best_val_cer": round(float(best["val_cer"]), 4),
               "best_val_wer": round(float(best["val_wer"]), 4), "best_epoch": int(best["epoch"]),
               "epochs_run": len(hist), "minutes": round(sum(float(h["secs"]) for h in hist) / 60, 2),
               "parameters": count_parameters(CRNN(charset.num_classes, mc))}
        new = not os.path.exists(csv_path)
        with open(csv_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=FIELDS)
            if new:
                w.writeheader()
            w.writerow(row)
        rows.append(row)
        el = time.time() - t_start
        if verbose:
            print(f"  [{i}/{len(todo)}] {name:28s} val CER {row['best_val_cer']:.4f}  WER {row['best_val_wer']:.4f}  "
                  f"({(time.time() - t0) / 60:.1f} min; ~{el / i * (len(todo) - i) / 60:.0f} min left)")
    return rank(rows)


def rank(rows):
    """Best first: lowest val CER, then lowest val WER, then name (deterministic)."""
    return sorted(rows, key=lambda r: (r["best_val_cer"], r["best_val_wer"], r["name"]))


def choose_settings(csv_path, defaults=DEFAULTS, useless_cer=0.99):
    """Settings for the main run: the sweep winner if there is a usable sweep, else the defaults.

    Returns (settings, info). info["source"] is "sweep" or "defaults", info["reason"] says why,
    info["winner"] is the chosen sweep row (or None). Falls back to the defaults when the CSV
    is missing/empty, or when no run learned anything (best val CER >= `useless_cer`, e.g. a
    too-short sweep), since then the ranking is noise."""
    rows = rank(load_results(csv_path)) if csv_path else []
    if not rows:
        return dict(defaults), {"source": "defaults", "winner": None, "n_runs": 0,
                                "reason": "no sweep results found, so using the default settings"}
    best = rows[0]
    if best["best_val_cer"] >= useless_cer:
        return dict(defaults), {"source": "defaults", "winner": None, "n_runs": len(rows),
                                "reason": f"no sweep run got below {useless_cer:.0%} val CER (ranking would be noise), "
                                          "so using the default settings"}
    chosen = {k: best[k] for k in SWEPT}
    return chosen, {"source": "sweep", "winner": best, "n_runs": len(rows),
                    "reason": f"best of {len(rows)} sweep runs by validation CER: {best['name']} "
                              f"(val CER {best['best_val_cer']:.4f}, WER {best['best_val_wer']:.4f} after "
                              f"{best['epochs']} epochs on {best['train_frac']:.0%} of train)"}


def make_configs(settings, **train_kw):
    """ModelConfig + TrainConfig for the chosen settings (other TrainConfig fields via train_kw)."""
    mc = ModelConfig(lstm_hidden=settings["lstm_hidden"], lstm_layers=settings["lstm_layers"],
                     dropout=settings["dropout"])
    tc = TrainConfig(lr=settings["lr"], batch_size=settings["batch_size"], **train_kw)
    return mc, tc


class RunMismatch(RuntimeError):
    pass


def check_run_dir(run_dir, model_cfg, train_cfg):
    """Refuse to resume a checkpoint that was trained with different settings.

    fit() resumes from run_dir/last.pt; if config.json there has another model or different
    lr / batch / weight decay / train fraction, resuming would silently mix two settings.
    (A different number of epochs is fine: that just trains longer.)"""
    p = Path(run_dir) / "config.json"
    if not p.exists():
        return "new run"
    old = json.loads(p.read_text())
    want_model = {"cnn_channels": list(model_cfg.cnn_channels), "lstm_hidden": model_cfg.lstm_hidden,
                  "lstm_layers": model_cfg.lstm_layers, "dropout": model_cfg.dropout}
    have_model = {k: (list(v) if isinstance(v, (list, tuple)) else v) for k, v in old.get("model", {}).items()}
    diffs = [f"model.{k}: checkpoint {have_model.get(k)!r} vs now {v!r}" for k, v in want_model.items()
             if have_model.get(k) != v]
    for k in ("lr", "batch_size", "weight_decay", "train_frac"):
        have = old.get("train", {}).get(k)
        now = getattr(train_cfg, k)
        if have is None or not math.isclose(float(have), float(now)):
            diffs.append(f"train.{k}: checkpoint {have} vs now {now}")
    if diffs:
        raise RunMismatch(f"{run_dir} was trained with different settings, so it can't be resumed:\n  "
                          + "\n  ".join(diffs) + "\nUse a new RUN_NAME (or delete that folder to start over).")
    return "resuming a run with the same settings" if (Path(run_dir) / "last.pt").exists() else "same settings"

### Plots

In [ ]:
%%writefile /content/aps360_htr/src/utils/plots.py
"""Plots for the report: training curves and example predictions."""
import csv

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def plot_history(history_csv, out_png):
    rows = list(csv.DictReader(open(history_csv)))
    ep = [int(r["epoch"]) for r in rows]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.plot(ep, [float(r["train_loss"]) for r in rows], label="train")
    a1.plot(ep, [float(r["val_loss"]) for r in rows], label="val")
    a1.set_xlabel("epoch"); a1.set_ylabel("CTC loss"); a1.set_title("Loss"); a1.legend()
    a2.plot(ep, [float(r["val_cer"]) for r in rows], label="val CER")
    a2.plot(ep, [float(r["val_wer"]) for r in rows], label="val WER")
    a2.set_xlabel("epoch"); a2.set_ylabel("error rate"); a2.set_title("Validation error"); a2.legend()
    a2.set_ylim(0, max(1.0, max(float(r["val_wer"]) for r in rows)) if rows else 1)
    fig.tight_layout(); fig.savefig(out_png, dpi=120); plt.close(fig)
    return out_png


def plot_predictions(images, refs, hyps, out_png, n=12):
    """Images with 'truth -> prediction' titles; green if right, red if wrong."""
    n = min(n, len(refs))
    fig, axes = plt.subplots(n, 1, figsize=(6, 0.85 * n + 0.4))
    axes = [axes] if n == 1 else axes
    for ax, img, r, h in zip(axes, images[:n], refs[:n], hyps[:n]):
        ax.imshow(img[0].numpy(), cmap="gray", vmin=-1, vmax=1)
        ax.set_title(f"{r!r} -> {h!r}", fontsize=8, loc="left", color="green" if r == h else "red")
        ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout(); fig.savefig(out_png, dpi=120); plt.close(fig)
    return out_png

### Report outputs (figures, LaTeX tables, numbers)
Shared with Step 3. `style.py` sizes every figure for the ICLR text width; `tables.py` writes CSV + booktabs `.tex`; `crnn_report.py` makes each Step 2 output.

In [ ]:
%%writefile /content/aps360_htr/src/report/__init__.py
"""Everything the progress report needs: figures, LaTeX tables, error analysis and key numbers.

All outputs go to one folder (outputs/report/ on Drive), separate from the training checkpoints.
"""

In [ ]:
%%writefile /content/aps360_htr/src/report/style.py
"""One look for every report figure, sized for the ICLR template.

The ICLR text block is 5.5 in wide (single column), so figures are drawn at
that width (or half of it) with 8-9 pt fonts: they stay readable when included
with \\includegraphics[width=\\linewidth]{...}. Each figure is saved twice:
PNG at 300 dpi (for previews / Word) and PDF (vector, best for LaTeX).
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FULL_WIDTH = 5.5    # inches: ICLR \textwidth
HALF_WIDTH = 2.7    # two figures side by side
GREEN, RED = "#1a7f37", "#c62828"
BLUE, ORANGE, GREY = "#1f77b4", "#ff7f0e", "#555555"

plt.rcParams.update({
    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5, "lines.linewidth": 1.4,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.03, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save_figure(fig, out_dir, name):
    """Save `fig` as <name>.png (300 dpi) and <name>.pdf. Returns both paths."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    paths = [out_dir / f"{name}.png", out_dir / f"{name}.pdf"]
    fig.savefig(paths[0], dpi=300)
    fig.savefig(paths[1])
    plt.close(fig)
    return [str(p) for p in paths]

In [ ]:
%%writefile /content/aps360_htr/src/report/tables.py
"""Write a table as CSV and as a LaTeX booktabs snippet you can \\input{} in Overleaf.

The .tex file is a complete `table` environment (caption + label), so in the report:
    \\usepackage{booktabs}          % in the preamble, once
    \\input{tables/table_metrics}   % where the table should go
"""
import csv
from pathlib import Path

_LATEX_SPECIAL = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
                  "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}


# Unicode that pdfLaTeX (Overleaf's default) can't typeset directly -> LaTeX commands.
_LATEX_UNICODE = {"×": r"\ensuremath{\times}", "→": r"\ensuremath{\rightarrow}", "∅": r"\ensuremath{\emptyset}",
                  "÷": r"\ensuremath{\div}", "≤": r"\ensuremath{\leq}", "≥": r"\ensuremath{\geq}",
                  "µ": r"\ensuremath{\mu}", "±": r"\ensuremath{\pm}", "–": "--", "—": "---",
                  "\u2018": "`", "\u2019": "'", "\u201c": "``", "\u201d": "''"}


def latex_escape(s) -> str:
    """Escape LaTeX specials (& % $ # _ { } ~ ^ \\) and convert non-ASCII symbols, so any text
    (including IAM labels such as '&' or '#') compiles with pdfLaTeX."""
    out = []
    for ch in str(s):
        if ch in _LATEX_SPECIAL:
            out.append(_LATEX_SPECIAL[ch])
        elif ch in _LATEX_UNICODE:
            out.append(_LATEX_UNICODE[ch])
        elif ch == '"':
            out.append(r"{\ttfamily\char34}")
        elif ch in "<>|":
            out.append(r"\ensuremath{" + ch + "}")
        else:
            out.append(ch)
    return "".join(out)


def fmt(v, digits=4):
    """Numbers -> short strings: ints with thousands separators, floats rounded."""
    if isinstance(v, bool) or v is None:
        return "" if v is None else str(v)
    if isinstance(v, int):
        return f"{v:,}"
    if isinstance(v, float):
        return f"{v:.{digits}f}"
    return str(v)


def to_latex(rows, columns, caption, label, col_format=None, raw_cols=(), note=None, digits=4) -> str:
    """booktabs table. `raw_cols` are columns whose cells are already LaTeX (not escaped)."""
    col_format = col_format or ("l" + "r" * (len(columns) - 1))
    lines = [r"\begin{table}[t]", r"\centering", r"\small",
             rf"\caption{{{latex_escape(caption)}}}", rf"\label{{{label}}}",
             rf"\begin{{tabular}}{{{col_format}}}", r"\toprule",
             " & ".join(rf"\textbf{{{latex_escape(c)}}}" for c in columns) + r" \\", r"\midrule"]
    for r in rows:
        if r == "midrule":
            lines.append(r"\midrule"); continue
        cells = [fmt(r.get(c, ""), digits) for c in columns]
        cells = [cell if c in raw_cols else latex_escape(cell) for c, cell in zip(columns, cells)]
        if r.get("_bold"):                      # highlight a row (e.g. the chosen sweep setting)
            cells = [rf"\textbf{{{c}}}" if c else c for c in cells]
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    if note:
        lines.append(rf"\par\smallskip{{\footnotesize {latex_escape(note)}\par}}")
    lines.append(r"\end{table}")
    return "\n".join(lines) + "\n"


def write_table(rows, columns, out_dir, name, caption, label=None, **kw):
    """Save <name>.csv and <name>.tex. `rows` is a list of dicts (or "midrule"). Returns both paths."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    csv_path, tex_path = out_dir / f"{name}.csv", out_dir / f"{name}.tex"
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        w.writeheader()
        w.writerows(r for r in rows if r != "midrule")
    tex_path.write_text(to_latex(rows, columns, caption, label or f"tab:{name}", **kw))
    return [str(csv_path), str(tex_path)]


def csv_to_markdown(csv_path) -> str:
    """A saved CSV table as a Markdown table (to preview it in the notebook)."""
    with open(csv_path, newline="") as f:
        rows = list(csv.reader(f))
    if not rows:
        return ""
    cell = lambda c: c.replace("|", "\\|")
    head = "| " + " | ".join(map(cell, rows[0])) + " |\n|" + "---|" * len(rows[0])
    return head + "\n" + "\n".join("| " + " | ".join(map(cell, r)) + " |" for r in rows[1:])

In [ ]:
%%writefile /content/aps360_htr/src/report/errors.py
"""Error analysis: which characters get confused, and how CER depends on word length."""
from collections import Counter

from src.utils.metrics import cer, wer

EMPTY = "\u2205"   # the symbol used for "nothing" in insertions / deletions


def align(ref: str, hyp: str):
    """Levenshtein alignment of two strings. Returns a list of (op, ref_char, hyp_char)
    with op in {"=", "sub", "del", "ins"}; del = character missed, ins = extra character."""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:   # walk back from the corner
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("=" if ref[i - 1] == hyp[j - 1] else "sub", ref[i - 1], hyp[j - 1])); i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(("del", ref[i - 1], EMPTY)); i -= 1
        else:
            ops.append(("ins", EMPTY, hyp[j - 1])); j -= 1
    return ops[::-1]


def char_confusions(refs, hyps):
    """Counter of (true char, predicted char) over all errors; EMPTY marks a missed/extra char."""
    c = Counter()
    for r, h in zip(refs, hyps):
        if r != h:
            c.update((a, b) for op, a, b in align(r, h) if op != "=")
    return c


def error_breakdown(refs, hyps):
    """Total substitutions / deletions / insertions (the three parts of CER)."""
    c = Counter()
    for r, h in zip(refs, hyps):
        if r != h:
            c.update(op for op, _, _ in align(r, h) if op != "=")
    return {"substitutions": c["sub"], "deletions": c["del"], "insertions": c["ins"],
            "reference_chars": sum(len(r) for r in refs)}


def word_mistakes(refs, hyps, k=15):
    """Most frequent (truth, prediction) pairs among wrong words."""
    return Counter((r, h) for r, h in zip(refs, hyps) if r != h).most_common(k)


LENGTH_BINS = [(1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 9), (10, 99)]


def bin_name(lo, hi):
    return str(lo) if lo == hi else (f"{lo}+" if hi >= 99 else f"{lo}-{hi}")


def cer_by_length(refs, hyps, bins=LENGTH_BINS):
    """CER and WER per word-length bin. Returns rows with bin, n, cer, wer."""
    rows = []
    for lo, hi in bins:
        pairs = [(r, h) for r, h in zip(refs, hyps) if lo <= len(r) <= hi]
        if pairs:
            rs, hs = zip(*pairs)
            rows.append({"length": bin_name(lo, hi), "n": len(pairs), "cer": cer(rs, hs), "wer": wer(rs, hs)})
    return rows

In [ ]:
%%writefile /content/aps360_htr/src/report/figures.py
"""Report figures. Every function returns the saved [png, pdf] paths (see style.save_figure)."""
import csv
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from src.report.style import (FULL_WIDTH, HALF_WIDTH, GREEN, RED, BLUE, ORANGE, GREY, save_figure)


def read_history(history_csv):
    """history.csv written by fit() (one row per epoch, survives resumes)."""
    rows = list(csv.DictReader(open(history_csv)))
    return [{k: float(v) for k, v in r.items()} for r in rows]


def _best_epoch(hist):
    return int(min(hist, key=lambda r: r["val_cer"])["epoch"])


def _loss_axes(ax, hist):
    ep = [r["epoch"] for r in hist]
    ax.plot(ep, [r["train_loss"] for r in hist], color=BLUE, label="train")
    ax.plot(ep, [r["val_loss"] for r in hist], color=ORANGE, label="validation")
    ax.axvline(_best_epoch(hist), color=GREY, ls=":", lw=1, label="best epoch")
    ax.set_xlabel("epoch"); ax.set_ylabel("CTC loss"); ax.set_title("Training and validation loss")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="none", loc="best")


def _error_axes(ax, hist):
    ep = [r["epoch"] for r in hist]
    ax.plot(ep, [r["val_cer"] for r in hist], color=BLUE, marker="o", ms=2.5, label="val CER")
    ax.plot(ep, [r["val_wer"] for r in hist], color=ORANGE, marker="s", ms=2.5, label="val WER")
    b = _best_epoch(hist)
    best = next(r for r in hist if int(r["epoch"]) == b)
    ax.axvline(b, color=GREY, ls=":", lw=1, label=f"best: epoch {b}, CER {best['val_cer']:.3f}")
    ax.set_ylim(0, min(1.05, max(max(r["val_wer"] for r in hist) * 1.1, 0.1)))
    ax.set_xlabel("epoch"); ax.set_ylabel("error rate"); ax.set_title("Validation CER and WER")
    ax.legend(frameon=True, framealpha=0.9, edgecolor="none", loc="best")


def _int_epochs(*axes):
    from matplotlib.ticker import MaxNLocator
    for ax in axes:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def plot_training_curves(history_csv, out_dir):
    """Three files: combined two-panel figure, loss only, CER/WER only."""
    hist = read_history(history_csv)
    paths = []
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(FULL_WIDTH, 2.1))
    _loss_axes(a1, hist); _error_axes(a2, hist); _int_epochs(a1, a2); fig.tight_layout()
    paths += save_figure(fig, out_dir, "fig_training_curves")
    fig, ax = plt.subplots(figsize=(HALF_WIDTH, 2.0)); _loss_axes(ax, hist); _int_epochs(ax); fig.tight_layout()
    paths += save_figure(fig, out_dir, "fig_loss_curve")
    fig, ax = plt.subplots(figsize=(HALF_WIDTH, 2.0)); _error_axes(ax, hist); _int_epochs(ax); fig.tight_layout()
    paths += save_figure(fig, out_dir, "fig_val_cer_wer_curve")
    return paths


def plot_prediction_grid(images, refs, hyps, out_dir, name, title=None, cols=3):
    """Word images (original resolution, uint8 arrays) with true and predicted text.
    Prediction is green when exactly right and red when wrong."""
    n = len(images)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(FULL_WIDTH, 0.95 * rows + (0.25 if title else 0)))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, img, r, h in zip(np.atleast_1d(axes).ravel(), images, refs, hyps):
        ax.axis("on"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(True); s.set_color("#cccccc"); s.set_linewidth(0.5)
        ax.imshow(img, cmap="gray", vmin=0, vmax=255, aspect="equal")
        ax.set_title(f"true: {r}", fontsize=7.5, loc="left", pad=2, parse_math=False)
        ax.set_xlabel(f"pred: {h if h else '(empty)'}", fontsize=7.5, loc="left", labelpad=2,
                      color=GREEN if r == h else RED, fontweight="bold", parse_math=False)
    if title:
        fig.suptitle(title, fontsize=9)
    fig.tight_layout(h_pad=0.6, w_pad=0.8)
    return save_figure(fig, out_dir, name)


def plot_char_confusions(confusions: Counter, out_dir, name="fig_char_confusions", k=15, title=None):
    """Horizontal bars: the k most frequent character errors (true -> predicted)."""
    top = confusions.most_common(k)[::-1]
    fig, ax = plt.subplots(figsize=(HALF_WIDTH, 0.17 * max(len(top), 1) + 0.6))
    labels = [f"{a!s} \u2192 {b!s}" for (a, b), _ in top]
    ax.barh(range(len(top)), [n for _, n in top], color=BLUE)
    ax.set_yticks(range(len(top)), labels, fontfamily="DejaVu Sans Mono")
    for t in ax.get_yticklabels():
        t.set_parse_math(False) if hasattr(t, "set_parse_math") else None
    ax.set_xlabel("count"); ax.grid(axis="y", visible=False)
    ax.set_title(title or "Most common character errors", loc="left")
    fig.tight_layout()
    return save_figure(fig, out_dir, name)


def plot_label_length_hist(texts, out_dir, name="fig_label_length_hist", split="train"):
    lengths = [len(t) for t in texts]
    c = Counter(lengths)
    xs = list(range(1, max(lengths) + 1))
    fig, ax = plt.subplots(figsize=(HALF_WIDTH, 1.9))
    ax.bar(xs, [c[x] for x in xs], color=BLUE, width=0.85)
    m = float(np.mean(lengths))
    ax.axvline(m, color=RED, ls="--", lw=1, label=f"mean {m:.1f}")
    ax.set_xlabel("word length (characters)"); ax.set_ylabel("words")
    ax.set_title(f"Label lengths ({split}, n={len(lengths):,})", loc="left")
    ax.legend(frameon=False); ax.grid(axis="x", visible=False)
    fig.tight_layout()
    return save_figure(fig, out_dir, name)


def plot_cer_by_length(systems: dict, out_dir, name="fig_cer_by_length", title=None):
    """Grouped bars of CER per word-length bin. `systems` maps a name to rows from
    errors.cer_by_length (all computed on the same words)."""
    names = list(systems)
    bins = [r["length"] for r in systems[names[0]]]
    ns = {r["length"]: r["n"] for r in systems[names[0]]}
    x = np.arange(len(bins)); w = 0.8 / len(names)
    fig, ax = plt.subplots(figsize=(FULL_WIDTH if len(names) > 1 else HALF_WIDTH * 1.25, 2.0))
    colors = [BLUE, ORANGE, GREEN, RED]
    for i, s in enumerate(names):
        vals = {r["length"]: r["cer"] for r in systems[s]}
        ax.bar(x + (i - (len(names) - 1) / 2) * w, [vals.get(b, 0) for b in bins], w,
               label=s, color=colors[i % len(colors)])
    ax.set_xticks(x, [f"{b}\n({ns[b]:,})" for b in bins], fontsize=6.5)
    ax.set_xlabel("word length in characters (number of test words)")
    ax.set_ylabel("CER"); ax.grid(axis="x", visible=False)
    ax.set_title(title or "CER by word length (test)", loc="left")
    if len(names) > 1:
        top = max(r["cer"] for rows in systems.values() for r in rows)
        ax.set_ylim(0, top * 1.28)       # headroom so the legend sits above the bars
        ax.legend(frameon=False, ncol=len(names), loc="upper right")
    fig.tight_layout()
    return save_figure(fig, out_dir, name)


# ------------------------------------------------------------------ block diagrams

def _box(ax, x, y, w, h, title, sub, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.015",
                                fc=color, ec="#333333", lw=0.6))
    ax.text(x + w / 2, y + h - 0.04, title, ha="center", va="top", fontsize=6.6, fontweight="bold")
    ax.text(x + w / 2, y + (h - 0.1) / 2, sub, ha="center", va="center", fontsize=5.9, color="#222222",
            linespacing=1.25)


def _arrow(ax, x0, y0, x1, y1):
    ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="-|>", lw=0.7, color="#333333",
                                                         shrinkA=0, shrinkB=0, mutation_scale=7))


def _row(ax, boxes, y, h, x0=0.005, x1=0.995, gap=0.022):
    w = (x1 - x0 - gap * (len(boxes) - 1)) / len(boxes)
    xs = [x0 + i * (w + gap) for i in range(len(boxes))]
    for x, (t, s, c) in zip(xs, boxes):
        _box(ax, x, y, w, h, t, s, c)
    for i in range(len(boxes) - 1):
        _arrow(ax, xs[i] + w, y + h / 2, xs[i + 1], y + h / 2)
    return xs, w


def crnn_shapes(model_cfg, num_classes, height=32):
    """Tensor shape after each stage, as text, computed from the model settings."""
    c = list(model_cfg.cnn_channels)
    pools = [(2, 2), (2, 2), (2, 1), (2, 1), (2, 1)]
    h, wdiv, out = height, 1, []
    for ch, (ph, pw) in zip(c, pools):
        h //= ph; wdiv *= pw
        out.append((ch, h, wdiv))
    return out


def draw_crnn_diagram(model_cfg, num_classes, out_dir, name="fig_crnn_architecture", height=32):
    """Block diagram of the CRNN with the tensor shape after every stage (W = image width)."""
    shapes = crnn_shapes(model_cfg, num_classes, height)
    pools = ["2×2", "2×2", "2×1", "2×1", "2×1"]
    wtxt = lambda d: "W" if d == 1 else f"W/{d}"
    conv = "#cfe2f3"; seq = "#d9ead3"; out = "#fce5cd"; inp = "#eeeeee"
    row1 = [("Input", f"grayscale\nimage\n1×{height}×W", inp)] + [
        (f"Conv {i + 1}", f"{ch} filters\npool {p}\n→ {ch}×{h}×{wtxt(d)}", conv)
        for i, ((ch, h, d), p) in enumerate(zip(shapes, pools))]
    T = wtxt(shapes[-1][2])
    H2 = 2 * model_cfg.lstm_hidden
    row2 = [("Reshape", f"to a sequence\nT = {T} steps\n→ T×{shapes[-1][0]}", seq),
            ("BiLSTM", f"{model_cfg.lstm_layers} layers, {model_cfg.lstm_hidden}\nper direction\n→ T×{H2}", seq),
            ("Linear", f"{H2} → {num_classes}\nlog-softmax\n→ T×{num_classes}", out),
            ("CTC", "loss in training\ngreedy decoding\nat test time", out),
            ("Output text", "e.g.\n\"which\"", inp)]
    fig, ax = plt.subplots(figsize=(FULL_WIDTH, 2.2))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    h = 0.38
    xs1, w1 = _row(ax, row1, 0.53, h)
    xs2, w2 = _row(ax, row2, 0.02, h, x0=0.06, x1=0.94)
    xa, xb, ym = xs1[-1] + w1 / 2, xs2[0] + w2 / 2, 0.465   # elbow arrow: last conv block -> reshape
    ax.plot([xa, xa, xb], [0.53, ym, ym], color="#333333", lw=0.7)
    _arrow(ax, xb, ym, xb, 0.02 + h)
    ax.text(0.005, 0.995, f"Each conv block: 3×3 convolution, BatchNorm, ReLU, max-pool.  "
                          f"CNN: height {height} → 1, width ÷ {shapes[-1][2]}", fontsize=6.3, va="top", color=GREY)
    return save_figure(fig, out_dir, name)


def draw_tesseract_pipeline(out_dir, name="fig_tesseract_pipeline", border=16, psm=8, oem=1):
    """Block diagram of the Tesseract baseline."""
    g = "#eeeeee"; b = "#cfe2f3"; o = "#fce5cd"
    boxes = [("Word image", "IAM crop at\noriginal resolution\n(grayscale)", g),
             ("Preprocess", f"pad a {border}-px\nwhite border\non every side", b),
             ("Tesseract", f"LSTM engine\n--oem {oem} --psm {psm}\n(single word, eng)", b),
             ("Normalize", "strip whitespace\nand newlines,\nASCII quotes", o),
             ("Score", "CER / WER vs\nIAM label (same\ncode as CRNN)", o)]
    fig, ax = plt.subplots(figsize=(FULL_WIDTH, 0.95))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    _row(ax, boxes, 0.03, 0.94, gap=0.04)
    return save_figure(fig, out_dir, name)


def plot_preprocessing_examples(originals, processed, texts, out_dir, name="fig_preprocessing_examples", height=32):
    """Cleaned training samples: the original IAM crop (left) and the CRNN input (right):
    grayscale, resized to `height` px keeping the aspect ratio, normalized to [-1, 1]."""
    n = len(originals)
    fig, axes = plt.subplots(n, 2, figsize=(FULL_WIDTH, 0.62 * n + 0.3),
                             gridspec_kw={"width_ratios": [1.15, 1]})
    axes = np.atleast_2d(axes)
    for (a, b), o, p, t in zip(axes, originals, processed, texts):
        a.imshow(o, cmap="gray", vmin=0, vmax=255)
        a.set_ylabel(f"\"{t}\"", rotation=0, ha="right", va="center", fontsize=8, parse_math=False)
        b.imshow(p, cmap="gray", vmin=-1, vmax=1)
        a.set_xlabel(f"{o.shape[1]}×{o.shape[0]} px", fontsize=6.5, labelpad=1, color=GREY)
        b.set_xlabel(f"{p.shape[1]}×{p.shape[0]} px", fontsize=6.5, labelpad=1, color=GREY)
        for ax in (a, b):
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(True); s.set_color("#cccccc"); s.set_linewidth(0.5)
    axes[0, 0].set_title("Original IAM word crop", fontsize=8)
    axes[0, 1].set_title(f"CRNN input: {height} px high, aspect kept, values in [-1, 1]", fontsize=8)
    fig.tight_layout(h_pad=0.3, w_pad=1.0)
    return save_figure(fig, out_dir, name)


def plot_sweep(rows, out_dir, name="fig_sweep", chosen=None, title=None):
    """Best validation CER of every sweep run (best at the top); the chosen run is highlighted."""
    rows = sorted(rows, key=lambda r: (r["best_val_cer"], r["best_val_wer"], r["name"]))[::-1]
    labels = [f"lr {r['lr']:g}, h {r['lstm_hidden']}, L {r['lstm_layers']}, d {r['dropout']:g}, b {r['batch_size']}"
              for r in rows]
    vals = [r["best_val_cer"] for r in rows]
    colors = [ORANGE if r["name"] == chosen else BLUE for r in rows]
    fig, ax = plt.subplots(figsize=(FULL_WIDTH, 0.19 * len(rows) + 0.7))
    ax.barh(range(len(rows)), vals, color=colors)
    ax.set_yticks(range(len(rows)), labels, fontsize=6.5, fontfamily="DejaVu Sans Mono")
    for i, v in enumerate(vals):
        ax.text(v, i, f" {v:.3f}", va="center", fontsize=6.5)
    ax.set_xlim(0, max(vals) * 1.15 if vals else 1)
    ax.set_xlabel("best validation CER (lower is better)"); ax.grid(axis="y", visible=False)
    ep = rows[0]["epochs"] if rows else 0
    fr = rows[0]["train_frac"] if rows else 0
    ax.set_title(title or f"Hyperparameter sweep ({len(rows)} runs, {ep} epochs on {fr:.0%} of train; "
                          "orange = chosen)", loc="left")
    fig.tight_layout()
    return save_figure(fig, out_dir, name)


def draw_emnist_pipeline(out_dir, name="fig_emnist_pipeline"):
    """Block diagram of the EMNIST character-classifier baseline."""
    g = "#eeeeee"; b = "#cfe2f3"; o = "#fce5cd"
    boxes = [("Word", "IAM crop,\noriginal size", g),
             ("Binarize", "Otsu, then\ndeslant and\ncomponents", b),
             ("Split", "cut wide\nblobs at ink\nminima", b),
             ("Normalize", "28x28,\ncentered,\nEMNIST-like", b),
             ("Char CNN", "EMNIST only,\n62 classes", o),
             ("Join", "punctuation\nrules; same\nCER / WER", o)]
    fig, ax = plt.subplots(figsize=(FULL_WIDTH, 0.95))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    _row(ax, boxes, 0.03, 0.94, gap=0.03)
    return save_figure(fig, out_dir, name)


def plot_segmentation_examples(examples, out_dir, name="fig_emnist_segmentation"):
    """Each row: the word with its character boxes (red = letter crop, blue = punctuation rule),
    then every normalized 28x28 crop with the classifier's prediction under it.
    `examples`: list of dicts with gray, pieces, crops, preds (one char per piece), ref, hyp."""
    from matplotlib.patches import Rectangle
    n = len(examples)
    fig, axes = plt.subplots(n, 2, figsize=(FULL_WIDTH, 0.85 * n + 0.25), gridspec_kw={"width_ratios": [1, 1.5]})
    axes = np.atleast_2d(axes)
    for (a, b), ex in zip(axes, examples):
        a.imshow(ex["gray"], cmap="gray", vmin=0, vmax=255)
        for q in ex["pieces"]:
            a.add_patch(Rectangle((q["x0"] - 0.5, q["y0"] - 0.5), q["x1"] - q["x0"], q["y1"] - q["y0"], fill=False,
                                  lw=0.7, ec=RED if q["kind"] == "char" else BLUE))
        ok = ex["ref"] == ex["hyp"]
        a.set_title(f"true: {ex['ref']}   pred: {ex['hyp'] or '(empty)'}", fontsize=7, loc="left", pad=2,
                    color=GREEN if ok else RED, parse_math=False)
        crops = list(ex["crops"]) or [np.zeros((28, 28), np.uint8)]
        gap = np.full((28, 4), 90, np.uint8)
        strip = np.hstack([x for c in crops for x in (c, gap)][:-1])
        b.imshow(strip, cmap="gray", vmin=0, vmax=255)
        chars = [p for p, q in zip(ex["preds"], ex["pieces"]) if q["kind"] == "char"]
        b.set_xticks([16 + 32 * i - 2 for i in range(len(chars))], chars, fontsize=7)
        b.tick_params(length=0, pad=1)
        a.set_xticks([]); a.set_yticks([]); b.set_yticks([])
        for ax in (a, b):
            ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(True); s.set_color("#cccccc"); s.set_linewidth(0.5)
    axes[0, 1].set_title("normalized 28x28 crops and predicted characters", fontsize=7, loc="left", pad=2)
    fig.tight_layout(h_pad=0.4, w_pad=0.8)
    return save_figure(fig, out_dir, name)

In [ ]:
%%writefile /content/aps360_htr/src/report/model_summary.py
"""Model complexity for the report: layer-by-layer shapes and parameters, plus MACs per word."""
import torch

from src.models.crnn import count_parameters


def layer_table(model, height=32, width=128):
    """One row per stage of the CRNN, measured with a real forward pass on a 1×height×width image.
    Output shapes are written per image (batch dimension dropped); W is the input width."""
    rows, shapes = [], {}
    stages = [(f"conv{i + 1}", blk) for i, blk in enumerate(model.cnn)] + [("rnn", model.rnn), ("fc", model.fc)]
    hooks = [m.register_forward_hook(lambda m, i, o, k=k: shapes.__setitem__(k, tuple((o[0] if isinstance(o, tuple) else o).shape)))
             for k, m in stages]
    was_training = model.training
    model.eval()
    dev = next(model.parameters()).device
    with torch.no_grad():
        model(torch.zeros(1, 1, height, width, device=dev))
    for h in hooks:
        h.remove()
    model.train(was_training)

    def wexpr(w_out):
        d = width // w_out
        return "W" if d == 1 else f"W/{d}"

    rows.append({"layer": "Input", "details": "grayscale word image", "output_shape": f"1×{height}×W", "parameters": 0})
    for i, blk in enumerate(model.cnn):
        conv = blk[0]
        pool = [m for m in blk if isinstance(m, torch.nn.MaxPool2d)]
        p = f", max-pool {pool[0].kernel_size[0]}×{pool[0].kernel_size[1]}" if pool else ""
        _, c, h, w = shapes[f"conv{i + 1}"]
        rows.append({"layer": f"Conv block {i + 1}",
                     "details": f"3×3 conv {conv.in_channels}→{conv.out_channels}, BN, ReLU{p}",
                     "output_shape": f"{c}×{h}×{wexpr(w)}",
                     "parameters": count_parameters(blk)})
    T, _, f = shapes["rnn"]
    c_last = model.rnn.input_size
    rows.append({"layer": "Reshape", "details": "columns become time steps (T = W/4)",
                 "output_shape": f"T×{c_last}", "parameters": 0})
    rnn = model.rnn
    rows.append({"layer": "BiLSTM", "details": f"{rnn.num_layers} layers, {rnn.hidden_size} per direction, dropout {model.dropout.p:g}",
                 "output_shape": f"T×{f}", "parameters": count_parameters(rnn)})
    rows.append({"layer": "Linear + log-softmax", "details": f"{model.fc.in_features}→{model.fc.out_features} (78 chars + CTC blank)",
                 "output_shape": f"T×{model.fc.out_features}", "parameters": count_parameters(model.fc)})
    total = count_parameters(model)
    assert total == sum(r["parameters"] for r in rows), "layer table must add up to the model total"
    return rows, total


def macs_per_word(model, height=32, width=128):
    """Multiply-accumulates for one word image (analytical: convs, LSTM gates, linear layer).
    BatchNorm, ReLU, pooling and softmax are negligible and not counted."""
    macs, h, w = 0, height, width
    for blk in model.cnn:
        conv = blk[0]
        macs += conv.out_channels * conv.in_channels * 9 * h * w      # padding 1: output = input size
        for m in blk:
            if isinstance(m, torch.nn.MaxPool2d):
                h //= m.kernel_size[0]; w //= m.kernel_size[1]
    T, rnn = w, model.rnn
    for layer in range(rnn.num_layers):
        n_in = rnn.input_size if layer == 0 else 2 * rnn.hidden_size
        macs += 2 * T * 4 * rnn.hidden_size * (n_in + rnn.hidden_size)  # 2 directions, 4 gates
    macs += T * model.fc.in_features * model.fc.out_features
    return macs

In [ ]:
%%writefile /content/aps360_htr/src/report/predict.py
"""Predictions with word IDs (so CRNN and Tesseract can be compared on exactly the same words)."""
import csv

import numpy as np
import torch
from PIL import Image

from src.data.iam import collate_fn
from src.models.crnn import CRNN
from src.training.sampler import BucketBatchSampler


@torch.no_grad()
def predict_dataset(model, ds, charset, device, batch_size=128, amp=False):
    """Greedy-decode every word of `ds`, batched exactly like evaluate() (same buckets, same
    padding, so the same predictions). Returns rows in dataset order."""
    model.eval()
    out = {}
    for idx in BucketBatchSampler(ds.widths(), batch_size, shuffle=False):
        samples = [(i, ds[i]) for i in idx]
        idx = [i for i, smp in samples if smp is not None]     # unreadable images are skipped
        if not idx:
            continue
        batch = collate_fn([smp for _, smp in samples if smp is not None])
        with torch.autocast(device.type, enabled=amp and device.type == "cuda"):
            lp = model(batch["images"].to(device))
        best = lp.float().argmax(2).cpu()
        L = CRNN.output_lengths(batch["widths"])
        for b, i in enumerate(idx):
            it = ds.items[i]
            out[i] = {"word_id": it.word_id, "writer": it.writer or "", "ref": it.text,
                      "hyp": charset.ctc_greedy_decode(best[: L[b], b].tolist())}
    return [out[i] for i in sorted(out)]


def save_predictions(rows, path):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["word_id", "writer", "ref", "hyp", "correct"])
        w.writeheader()
        for r in rows:
            w.writerow({**r, "correct": int(r["ref"] == r["hyp"])})
    return str(path)


def load_predictions(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def original_image(item):
    """The word image at its original resolution (uint8, white background), for figures."""
    try:
        with Image.open(item.path) as im:
            return np.asarray(im.convert("L"))
    except Exception:
        return np.full((32, 64), 255, dtype=np.uint8)   # unreadable file: blank placeholder (figures only)

In [ ]:
%%writefile /content/aps360_htr/src/report/manifest.py
"""Keeps track of every report file: a caption per file (report_summary.md) and every key
number in one place (report_numbers.json). Steps 2 and 3 both add to the same folder."""
import json
import os
from pathlib import Path

_EXTRA_CAPTIONS = {
    "_val_predictions.csv": "{name}: every validation prediction with its word ID, writer, truth and prediction.",
    "_test_predictions.csv": "{name}: every test prediction with its word ID, writer, truth and prediction (used for the comparison).",
    "report_numbers.json": "Every key number from Steps 2-5 in one place (metrics, model size, training, data, comparison).",
    "report_summary.md": "This file: every report output with a suggested caption.",
    "manifest.json": "Internal list of files and captions used to write this summary.",
}

KINDS = {".png": "figure", ".pdf": "figure", ".csv": "table", ".tex": "table", ".json": "data", ".md": "data"}


class Report:
    def __init__(self, out_dir, step):
        self.dir = Path(out_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.step = step
        self.manifest_path = self.dir / "manifest.json"
        self.numbers_path = self.dir / "report_numbers.json"

    # -- captions
    def _manifest(self):
        return json.loads(self.manifest_path.read_text()) if self.manifest_path.exists() else {}

    def add(self, paths, caption):
        """Record the files just saved (e.g. [x.png, x.pdf]) with one suggested caption."""
        m = self._manifest()
        for p in paths:
            m[Path(p).name] = {"caption": caption, "step": self.step}
        self.manifest_path.write_text(json.dumps(m, indent=1, ensure_ascii=False))
        return paths

    # -- numbers
    def numbers(self, section=None, data=None):
        """Merge `data` into report_numbers.json under `section` (replacing that section)."""
        nums = json.loads(self.numbers_path.read_text()) if self.numbers_path.exists() else {}
        if section is not None:
            nums[section] = data
            self.numbers_path.write_text(json.dumps(nums, indent=2, ensure_ascii=False))
        return nums

    # -- summary
    def write_summary(self, future_work=None):
        m = self._manifest()
        for f in self.dir.iterdir():          # data files written outside rep.add()
            for pat, cap in _EXTRA_CAPTIONS.items():
                if f.is_file() and f.name not in m and f.name.endswith(pat):
                    who = f.name.split("_")[0]
                    m[f.name] = {"caption": cap.format(name={"crnn": "CRNN", "tesseract": "Tesseract",
                                                             "emnist": "EMNIST baseline"}.get(who, who)),
                                 "step": {"crnn": "2", "tesseract": "3", "emnist": "4"}.get(who, "2-4")}
        groups = {}
        for name, info in m.items():
            if not (self.dir / name).exists():
                continue
            stem, ext = os.path.splitext(name)
            kind = "data" if (ext == ".csv" and stem.endswith("_predictions")) else KINDS.get(ext, "data")
            groups.setdefault((kind, stem), {"exts": [], **info})["exts"].append(ext)
        lines = ["# Report outputs", "",
                 "Everything in this folder was generated by the notebooks (Step 2 = CRNN, Step 3 = Tesseract baseline, "
                 "Step 4 = EMNIST character-classifier baseline, Step 5 = webcam test set).",
                 "", "**Overleaf:** upload the folder, add `\\usepackage{booktabs}` and `\\usepackage{graphicx}` to the preamble, then",
                 "use `\\includegraphics[width=\\linewidth]{fig_name.pdf}` for figures and `\\input{table_name.tex}` for tables",
                 "(each `.tex` is a complete `table` with caption and label; edit the caption to taste).", ""]
        titles = {"figure": "## Figures (PNG 300 dpi + PDF)", "table": "## Tables (CSV + LaTeX booktabs)", "data": "## Data files"}
        for kind in ("figure", "table", "data"):
            items = sorted((s, g) for (k, s), g in groups.items() if k == kind)
            if not items:
                continue
            lines += [titles[kind], ""]
            for stem, g in items:
                files = ", ".join(f"`{stem}{e}`" for e in sorted(g["exts"]))
                lines.append(f"- {files} (step {g['step']}): {g['caption']}")
            lines.append("")
        nums = self.numbers()
        if nums:
            lines += ["## Key numbers", "", "Headline values (every number is in `report_numbers.json`):", ""]
            lines += [f"- {h}" for h in headlines(nums)] + [""]
        if future_work:
            lines += ["## Future work", ""] + [f"- {x}" for x in future_work] + [""]
        path = self.dir / "report_summary.md"
        path.write_text("\n".join(lines))
        return str(path)

    def list_files(self):
        files = sorted(p for p in self.dir.iterdir() if p.is_file())
        for p in files:
            print(f"  {p.stat().st_size / 1024:8.1f} KB  {p.name}")
        print(f"{len(files)} files in {self.dir}")
        return files


def _flatten(d, prefix=""):
    """{'test': {'cer': 0.1}} -> {'test.cer': 0.1}; skips lists and long nested blocks."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(_flatten(v, f"{prefix}{k}."))
        elif isinstance(v, (int, float, str, bool)) and v is not None:
            out[f"{prefix}{k}"] = v
    return out


def _short(v):
    if isinstance(v, bool):
        return str(v)
    return f"{v:.4f}" if isinstance(v, float) else (f"{v:,}" if isinstance(v, int) else str(v))


def headlines(nums):
    """Short, readable list of the numbers most likely to go into the report text."""
    out, p = [], lambda x: f"{100 * x:.2f}%"
    c = nums.get("crnn")
    if c:
        out.append(f"CRNN: val CER {p(c['val']['cer'])}, WER {p(c['val']['wer'])} ({c['val']['words']:,} words); "
                   f"test CER {p(c['test']['cer'])}, WER {p(c['test']['wer'])} ({c['test']['words']:,} words); "
                   f"{c['test']['ms_per_word_batched']:.2f} ms/word batched, {c['ms_single_word']:.1f} ms for one word on {c['device']}")
    t = nums.get("tesseract")
    if t:
        parts = [f"{s} CER {p(t[s]['cer'])}, WER {p(t[s]['wer'])} ({t[s]['words']:,} of {t[s]['of_words']:,} words)"
                 for s in ("val", "test") if s in t]
        ms = t.get("test", {}).get("ms_per_word_cpu_core")
        out.append(f"Tesseract {t.get('version', '')} ({t.get('config', '')}): " + "; ".join(parts)
                   + (f"; {ms:.0f} ms/word per CPU core" if ms else ""))
    m = nums.get("model")
    if m:
        out.append(f"Model: {m['parameters']:,} parameters ({m['size_mb_fp32']} MB float32), "
                   f"{m['macs_per_word_w128'] / 1e6:.0f} M MACs per 128-px word, {m['num_classes']} output classes")
    tr = nums.get("training")
    if tr:
        out.append(f"Training: {tr['epochs_run']} epochs run (max {tr['max_epochs']}), best epoch {tr['best_epoch']} "
                   f"(val CER {p(tr['best_val_cer'])}), {tr['train_minutes']:.1f} min on {tr['device']}")
    d = nums.get("data")
    if d:
        sp = d["splits"]
        out.append(f"Data ({d['label_source']}): " + ", ".join(
            f"{s} {sp[s]['words']:,} words / {sp[s]['writers']} writers" for s in ("train", "val", "test"))
            + f"; {d['writers_shared_between_splits']} writers shared between splits")
    sw = nums.get("sweep")
    if isinstance(sw, dict) and sw.get("runs"):
        best = sw["runs"][0]
        out.append(f"Sweep: {len(sw['runs'])} runs ({sw['epochs_per_run']} epochs on {sw['train_frac']:.0%} of train, "
                   f"{sw['total_minutes']:.0f} min total); best {best['name']} val CER {p(best['best_val_cer'])}; "
                   f"chosen: {sw.get('chosen') or 'defaults'}")
    e = nums.get("emnist")
    if e:
        parts = [f"{s} CER {p(e[s]['cer'])}, WER {p(e[s]['wer'])} (case-insensitive CER {p(e[s]['cer_case_insensitive'])}; "
                 f"{e[s]['words']:,} of {e[s]['of_words']:,} words)" for s in ("val", "test") if s in e]
        c = e.get("classifier", {})
        acc = c.get("test", {}).get("accuracy")
        out.append("EMNIST baseline: " + "; ".join(parts)
                   + (f"; classifier EMNIST test accuracy {p(acc)}" if acc is not None else "")
                   + (f"; {e['test']['ms_per_word']:.1f} ms/word" if "test" in e else ""))
    cmp = nums.get("comparison")
    if cmp:
        models = [k for k in cmp if isinstance(cmp[k], dict) and "test" in cmp[k]]
        if models:
            out.append("Comparison (same test words): " + "; ".join(
                f"{k} CER {p(cmp[k]['test']['cer'])} / WER {p(cmp[k]['test']['wer'])}" for k in models))
        if cmp.get("note"):
            out.append("Comparison: " + cmp["note"])
    wc = nums.get("webcam")
    if isinstance(wc, dict) and wc.get("models"):
        out.append(f"Webcam test set ({wc['words']} words): " + "; ".join(
            f"{v['name']} CER {p(v['cer'])} / WER {p(v['wer'])}" for v in wc["models"].values()))
    return out


FUTURE_WORK = {
    "emnist": "EMNIST character-classifier baseline (segment the word into characters, classify each with a CNN "
              "trained on EMNIST): planned second baseline.",
    "augment": "Webcam-style augmentation (blur, perspective, lighting, noise) and the with/without ablation.",
}


def future_work(rep):
    """Future-work items that are still open (EMNIST disappears once Step 4 has run, augmentation once Step 6 has)."""
    done = {"emnist": (rep.dir / "emnist_metrics.json").exists(),
            "augment": (rep.dir.parent / "model_b" / "model_b_metrics.json").exists()}
    return [v for k, v in FUTURE_WORK.items() if not done.get(k)]

In [ ]:
%%writefile /content/aps360_htr/src/report/crnn_report.py
"""Step 2: turn a finished (or resumed) CRNN run into report figures, tables and numbers.

Everything is rebuilt from files on Drive (history.csv, best.pt) plus the evaluation, so it
works the same whether training ran in one go or was resumed after a disconnect.
"""
import csv
import json
import random
from collections import Counter
from pathlib import Path

from src.report import errors, figures, tables
from src.report.model_summary import layer_table, macs_per_word
from src.report.predict import original_image
from src.utils.metrics import cer, wer

pct = lambda x: f"{100 * x:.2f}"


def save_curves(rep, history_csv):
    hist = figures.read_history(history_csv)
    paths = figures.plot_training_curves(history_csv, rep.dir)
    rep.add(paths[0:2], "Training (blue) and validation (orange) CTC loss, and validation CER/WER per epoch; "
                        "the dotted line marks the best epoch (lowest validation CER), whose weights are kept.")
    rep.add(paths[2:4], "Training and validation CTC loss per epoch.")
    rep.add(paths[4:6], "Validation CER and WER per epoch (greedy CTC decoding).")
    return paths, hist


def save_metrics(rep, preds, eval_results, latency_ms, device_name):
    """Final val/test metrics. `preds` maps split -> rows from predict_dataset."""
    rows, nums = [], {"device": device_name, "ms_single_word": round(latency_ms, 3)}
    for split in ("val", "test"):
        p = preds[split]
        refs, hyps = [r["ref"] for r in p], [r["hyp"] for r in p]
        c, w = cer(refs, hyps), wer(refs, hyps)
        ms = eval_results[split]["ms_per_word"]
        nums[split] = {"cer": c, "wer": w, "words": len(p), "word_accuracy": 1 - w, "ms_per_word_batched": ms,
                       "loss": eval_results[split]["loss"]}
        rows.append({"Split": split, "Words": len(p), "CER (%)": pct(c), "WER (%)": pct(w),
                     "ms/word (batch 128)": f"{ms:.2f}", "ms (1 word)": f"{latency_ms:.1f}"})
    rep.numbers("crnn", nums)
    (rep.dir / "crnn_metrics.json").write_text(json.dumps(nums, indent=2))
    rep.add([str(rep.dir / "crnn_metrics.json")], "CRNN val/test CER, WER and speed (read by Step 3 for the comparison).")
    paths = tables.write_table(rows, list(rows[0]), rep.dir, "table_crnn_metrics",
                               f"CRNN results on the IAM validation and test splits (writer-independent, greedy decoding). "
                               f"Speed on {device_name}.", "tab:crnn_metrics")
    rep.add(paths, "Final CRNN metrics: CER and WER on val and test, batched throughput and single-word latency.")
    return nums


def save_model_summary(rep, model, charset, ck, train_cfg, model_cfg, history, device_name, height=32,
                       settings_info=None):
    rows, total = layer_table(model, height=height)
    macs = macs_per_word(model, height=height, width=128)
    rows_out = rows + ["midrule", {"layer": "Total", "details": f"{total * 4 / 1e6:.1f} MB as float32",
                                   "output_shape": "", "parameters": total}]
    paths = tables.write_table(
        [r if r == "midrule" else {"Layer": r["layer"], "Details": r["details"], "Output shape": r["output_shape"],
                                    "Parameters": r["parameters"]} for r in rows_out],
        ["Layer", "Details", "Output shape", "Parameters"], rep.dir, "table_model_summary",
        f"CRNN architecture. Output shapes are per image (C×H×W, or T×features); W is the input width and "
        f"T = W/4 time steps. {total:,} trainable parameters, {macs / 1e6:.0f} M multiply-accumulates for a 128-px-wide word.",
        "tab:model_summary", col_format="lllr")
    rep.add(paths, "Layer-by-layer CRNN summary with output shapes and parameter counts (model complexity).")

    best_epoch = int(ck["epoch"])
    train_secs = sum(float(r["secs"]) for r in history)
    hp = [("Optimizer", f"AdamW (weight decay {train_cfg.weight_decay:g})"),
          ("Learning rate", f"{train_cfg.lr:g}, halved after {train_cfg.lr_patience} epochs without val-CER gain"),
          ("Scheduler", "ReduceLROnPlateau (factor 0.5) on validation CER"),
          ("Early stopping", f"patience {train_cfg.patience} epochs"),
          ("Batch size", train_cfg.batch_size), ("Gradient clipping", f"max norm {train_cfg.grad_clip:g}"),
          ("Mixed precision", "fp16 on GPU" if train_cfg.amp else "off"),
          ("Epochs run / best epoch", f"{len(history)} / {best_epoch}"),
          ("Training time", f"{train_secs / 60:.1f} min on {device_name}"),
          ("CNN channels", " / ".join(map(str, model_cfg.cnn_channels))),
          ("BiLSTM", f"{model_cfg.lstm_layers} layers, hidden {model_cfg.lstm_hidden} per direction"),
          ("Dropout", f"{model_cfg.dropout:g}"),
          ("Input height", f"{height} px (width kept in proportion, max 512)"),
          ("Output classes", f"{charset.num_classes} ({len(charset.chars)} characters + CTC blank)"),
          ("Loss / decoding", "CTC (blank = 0) / greedy best path"),
          ("Parameters", f"{total:,}")]
    if settings_info:
        w = settings_info.get("winner")
        hp.append(("Settings chosen by", f"hyperparameter sweep: best of {settings_info['n_runs']} runs by validation CER "
                                         f"({w['name']})" if settings_info["source"] == "sweep" and w
                                         else "defaults (" + settings_info["reason"] + ")"))
    hp_rows = [{"Hyperparameter": k, "Value": v} for k, v in hp]
    cap = "Training setup and hyperparameters of the CRNN" + (
        " (learning rate, LSTM size/depth, dropout and batch size chosen by the hyperparameter sweep; see the sweep table)."
        if settings_info and settings_info.get("source") == "sweep" else ".")
    paths = tables.write_table(hp_rows, ["Hyperparameter", "Value"], rep.dir, "table_hyperparameters",
                               cap, "tab:hyperparameters", col_format="lp{3.6in}")
    rep.add(paths, "Hyperparameters and training setup (optimizer, schedule, epochs run, best epoch, model size).")
    nums = {"parameters": total, "size_mb_fp32": round(total * 4 / 1e6, 2), "macs_per_word_w128": macs,
            "num_classes": charset.num_classes, "charset_size": len(charset.chars), "input_height": height,
            "cnn_channels": list(model_cfg.cnn_channels), "lstm_hidden": model_cfg.lstm_hidden,
            "lstm_layers": model_cfg.lstm_layers, "dropout": model_cfg.dropout,
            "layers": [{k: v for k, v in r.items()} for r in rows]}
    rep.numbers("model", nums)
    rep.numbers("training", {"optimizer": "AdamW", "lr": train_cfg.lr, "weight_decay": train_cfg.weight_decay,
                             "scheduler": "ReduceLROnPlateau(factor=0.5, patience=%d) on val CER" % train_cfg.lr_patience,
                             "early_stopping_patience": train_cfg.patience, "batch_size": train_cfg.batch_size,
                             "grad_clip": train_cfg.grad_clip, "amp": train_cfg.amp, "max_epochs": train_cfg.epochs,
                             "epochs_run": len(history), "best_epoch": best_epoch,
                             "best_val_cer": float(ck["best_cer"]), "train_minutes": round(train_secs / 60, 2),
                             "device": device_name,
                             "settings_source": (settings_info or {}).get("source", "unknown"),
                             "settings_reason": (settings_info or {}).get("reason", "")})
    return rows, total, macs


def save_data_stats(rep, stats, datasets, label_source):
    f_rows = [("words.txt lines", stats.get("total_lines", 0)),
              ("removed: segmentation marked 'err'", stats.get("segmentation_err", 0)),
              ("removed: characters outside the charset", stats.get("chars_outside_charset", 0)),
              ("removed: label longer than 32", stats.get("too_long", 0)),
              ("removed: empty label", stats.get("empty_label", 0)),
              ("removed: missing or broken image", stats.get("bad_or_missing_image", 0)),
              ("removed: cut-off words.txt line", stats.get("malformed_line", 0)),
              ("kept (pass all filters)", stats.get("kept", 0)),
              ("not in any Aachen split", stats.get("not_in_any_split", 0))]
    paths = tables.write_table([{"Step": k, "Words": v} for k, v in f_rows], ["Step", "Words"], rep.dir,
                               "table_data_filtering", f"IAM word filtering (labels: {label_source}).",
                               "tab:data_filtering", col_format="lr")
    rep.add(paths, "How many IAM words each filter removed, and how many remain.")
    s_rows, split_nums = [], {}
    for split in ("train", "val", "test"):
        items = datasets[split].items
        writers = {it.writer for it in items if it.writer}
        forms = {it.form_id for it in items}
        mean_len = sum(len(it.text) for it in items) / max(len(items), 1)
        s_rows.append({"Split": split, "Words": len(items), "Writers": len(writers), "Forms": len(forms),
                       "Mean length": f"{mean_len:.2f}"})
        split_nums[split] = {"words": len(items), "writers": len(writers), "forms": len(forms), "mean_label_len": mean_len}
    shared = 0
    ws = {s: {it.writer for it in datasets[s].items if it.writer} for s in ("train", "val", "test")}
    shared = len((ws["train"] & ws["val"]) | (ws["train"] & ws["test"]) | (ws["val"] & ws["test"]))
    paths = tables.write_table(s_rows, list(s_rows[0]), rep.dir, "table_data_splits",
                               f"Aachen writer-independent splits after filtering; {shared} writers appear in more than one split.",
                               "tab:data_splits")
    rep.add(paths, "Words, writers and forms per split (writer-disjoint Aachen split) and mean label length.")
    rep.numbers("data", {"label_source": label_source, "filtering": {k: v for k, v in f_rows},
                         "splits": split_nums, "writers_shared_between_splits": shared,
                         "writer_source": stats.get("writer_source"), "split_method": stats.get("split_method")})
    return s_rows


def save_label_hist(rep, datasets):
    paths = figures.plot_label_length_hist([it.text for it in datasets["train"].items], rep.dir)
    rep.add(paths, "Distribution of label lengths in the training split; most IAM words are short.")
    return paths


def save_preprocessing_examples(rep, ds, n=4, seed=3):
    """At least one cleaned training sample (rubric): original crop vs. the CRNN's 32-px input."""
    rng = random.Random(seed)
    idx = [i for i in rng.sample(range(len(ds)), min(len(ds), 50)) if 3 <= len(ds.items[i].text) <= 10][:n]
    originals = [original_image(ds.items[i]) for i in idx]
    processed = [ds[i][0][0].numpy() for i in idx]
    texts = [ds.items[i].text for i in idx]
    paths = figures.plot_preprocessing_examples(originals, processed, texts, rep.dir)
    rep.add(paths, "Cleaned training samples: the original IAM word crop (left) and the CRNN input (right): grayscale, "
                   "resized to 32 px high keeping the aspect ratio, normalized to [-1, 1]; the label is shown on the left.")
    return paths


def save_prediction_grid(rep, ds, preds, n=12, seed=0, split="test", name="fig_crnn_predictions"):
    """A random (not cherry-picked) sample of words with true and predicted text."""
    idx = sorted(random.Random(seed).sample(range(len(preds)), min(n, len(preds))))
    by_id = {it.word_id: it for it in ds.items}
    rows = [preds[i] for i in idx]
    imgs = [original_image(by_id[r["word_id"]]) for r in rows]
    right = sum(r["ref"] == r["hyp"] for r in rows)
    paths = figures.plot_prediction_grid(imgs, [r["ref"] for r in rows], [r["hyp"] for r in rows], rep.dir, name)
    rep.add(paths, f"CRNN predictions on {len(rows)} randomly chosen {split} words from unseen writers "
                   f"({right}/{len(rows)} exactly right); green = correct, red = wrong.")
    return paths


def save_error_analysis(rep, preds, system="CRNN", prefix="crnn", k=15):
    refs, hyps = [r["ref"] for r in preds], [r["hyp"] for r in preds]
    conf = errors.char_confusions(refs, hyps)
    bd = errors.error_breakdown(refs, hyps)
    subs = Counter({pair: n for pair, n in conf.items() if errors.EMPTY not in pair})
    paths = figures.plot_char_confusions(subs, rep.dir, f"fig_{prefix}_char_confusions", k=k,
                                         title=f"{system}: top substitutions (test)\n"
                                               f"{bd['substitutions']:,} sub. / {bd['deletions']:,} del. / {bd['insertions']:,} ins.")
    rep.add(paths, f"{system}: the {k} most frequent character substitutions (true → predicted) on the test split; "
                   "the title gives the total substitutions, deletions and insertions behind its CER.")
    rows = [{"True": a, "Predicted": b, "Count": n} for (a, b), n in conf.most_common(k)]
    p2 = tables.write_table(rows, ["True", "Predicted", "Count"], rep.dir, f"table_{prefix}_char_confusions",
                            f"{system}: most frequent character errors on the test split (∅ = missing/extra character).",
                            f"tab:{prefix}_char_confusions", col_format="ccr")
    rep.add(p2, f"{system} top character confusions as a table.")
    wm = [{"True word": r, "Predicted": h if h else "(empty)", "Count": n} for (r, h), n in errors.word_mistakes(refs, hyps, k)]
    p3 = tables.write_table(wm, ["True word", "Predicted", "Count"], rep.dir, f"table_{prefix}_word_errors",
                            f"{system}: most frequent wrong words on the test split.", f"tab:{prefix}_word_errors",
                            col_format="llr")
    rep.add(p3, f"{system} most common whole-word mistakes (qualitative error analysis).")
    rep.numbers(f"{prefix}_error_breakdown", bd)
    return conf, bd, paths + p2 + p3


def save_cer_by_length(rep, preds, system="CRNN", prefix="crnn"):
    rows = errors.cer_by_length([r["ref"] for r in preds], [r["hyp"] for r in preds])
    paths = figures.plot_cer_by_length({system: rows}, rep.dir, f"fig_{prefix}_cer_by_length",
                                       title=f"{system}: CER by word length (test)")
    rep.add(paths, f"{system}: CER on the test split grouped by word length (number of words per group in brackets).")
    p2 = tables.write_table([{"Length": r["length"], "Words": r["n"], "CER (%)": pct(r["cer"]), "WER (%)": pct(r["wer"])}
                             for r in rows], ["Length", "Words", "CER (%)", "WER (%)"], rep.dir,
                            f"table_{prefix}_cer_by_length", f"{system} error rates by word length (test).",
                            f"tab:{prefix}_cer_by_length")
    rep.add(p2, f"{system} CER/WER per word-length group (numbers behind the bar chart).")
    rep.numbers(f"{prefix}_cer_by_length", rows)
    return paths + p2


def save_architecture(rep, model_cfg, num_classes):
    paths = figures.draw_crnn_diagram(model_cfg, num_classes, rep.dir)
    rep.add(paths, "CRNN architecture: five convolutional blocks reduce the 32-px-high image to one feature column "
                   "per 4 pixels of width, a 2-layer BiLSTM reads the sequence, and a linear layer outputs "
                   f"{num_classes} classes per time step for CTC.")
    return paths


def save_sweep(rep, sweep_csv, chosen_name=None):
    """table_sweep (all swept settings, best first, chosen row in bold) + fig_sweep."""
    from src.training.sweep import load_results, rank
    rows = rank(load_results(sweep_csv))
    if not rows:
        print("no sweep results (RUN_SWEEP was False and no sweep_results.csv): skipping the sweep table")
        return None
    out = []
    for i, r in enumerate(rows, 1):
        chosen = r["name"] == chosen_name
        out.append({"#": f"{i}*" if chosen else i, "LR": f"{r['lr']:g}", "Hidden": r["lstm_hidden"],
                    "Layers": r["lstm_layers"], "Dropout": f"{r['dropout']:g}", "Batch": r["batch_size"],
                    "Val CER (%)": pct(r["best_val_cer"]), "Val WER (%)": pct(r["best_val_wer"]),
                    "Best ep.": r["best_epoch"], "Min": f"{r['minutes']:.1f}",
                    "Chosen": "yes" if chosen else "", "_bold": chosen})
    ep, fr = rows[0]["epochs"], rows[0]["train_frac"]
    cols = ["#", "LR", "Hidden", "Layers", "Dropout", "Batch", "Val CER (%)", "Val WER (%)", "Best ep.", "Min"]
    note = "* = setting used for the final model." if chosen_name else "No run was chosen (defaults used)."
    paths = tables.write_table(out, cols + ["Chosen"], rep.dir, "table_sweep", "", "tab:sweep")
    # the .tex leaves out the Chosen column (the bold row and * mark it); the CSV keeps it
    short = {"Dropout": "Drop.", "Val CER (%)": "CER (%)", "Val WER (%)": "WER (%)", "Best ep.": "Ep."}
    tex_rows = [{short.get(k, k): v for k, v in r.items()} for r in out]
    tex = tables.to_latex(tex_rows, [short.get(c, c) for c in cols],
                          f"Hyperparameter sweep, ranked by validation CER (test set never used). Each run: {ep} epochs on "
                          f"{fr:.0%} of the training words; CER/WER on the full validation set; Ep. = best epoch; Min = minutes. A small grid over learning rate and LSTM "
                          "hidden size, plus one-at-a-time changes of dropout, batch size, hidden size and LSTM depth "
                          "around the default (lr 0.001, hidden 256, dropout 0.3, batch 64, 2 layers).",
                          "tab:sweep", note=note)
    Path(paths[1]).write_text(tex)
    rep.add(paths, f"Hyperparameter sweep ({len(rows)} runs, {ep} epochs on {fr:.0%} of train each), ranked by "
                   "validation CER; the bold row (*) is the setting used for the final model.")
    p2 = figures.plot_sweep(rows, rep.dir, chosen=chosen_name)
    rep.add(p2, "Best validation CER of every sweep run; the orange bar is the setting chosen for the final model.")
    rep.numbers("sweep", {"runs": rows, "chosen": chosen_name, "epochs_per_run": ep, "train_frac": fr,
                          "total_minutes": round(sum(r["minutes"] for r in rows), 1)})
    return paths + p2

In [ ]:
%%writefile /content/aps360_htr/src/report/compare.py
"""CRNN vs the baselines (Tesseract, EMNIST character classifier) on exactly the same words.

Needs crnn_metrics.json + crnn_<split>_predictions.csv (Step 2) and, for each baseline that has run,
<name>_metrics.json + <name>_<split>_predictions.csv (Step 3 = tesseract, Step 4 = emnist) in the
report folder. Whichever notebook runs last builds the table with every model available.
All models are scored on the words they **all** have (matched by word ID): if a baseline was run on
a random subset of the test set, everyone is scored on that subset, so the comparison is like-for-like.
"""
import json

from src.report import errors, figures, tables
from src.report.predict import load_predictions
from src.utils.metrics import cer, wer

pct = lambda x: f"{100 * x:.2f}"

SYSTEMS = [("CRNN (ours)", "crnn"), ("Tesseract (LSTM)", "tesseract"), ("EMNIST CNN + splitter", "emnist")]
BASELINES = SYSTEMS[1:]


def _scores(rows):
    refs, hyps = [r["ref"] for r in rows], [r["hyp"] for r in rows]
    return {"cer": cer(refs, hyps), "wer": wer(refs, hyps), "words": len(rows)}


def available(d):
    """{prefix: metrics dict} for every system whose metrics + test predictions exist."""
    out = {}
    for _, p in SYSTEMS:
        if (d / f"{p}_metrics.json").exists() and (d / f"{p}_test_predictions.csv").exists():
            out[p] = json.loads((d / f"{p}_metrics.json").read_text())
    return out


def _speed(prefix, m):
    if prefix == "crnn":
        return m.get("test", {}).get("ms_per_word_batched")
    if prefix == "tesseract":
        return m.get("test", {}).get("ms_per_word_cpu_core")
    return m.get("test", {}).get("ms_per_word")


def save_comparison(rep):
    d = rep.dir
    have = available(d)
    if "crnn" not in have or len(have) < 2:
        missing = ["Step 2 (CRNN)"] if "crnn" not in have else []
        if len(have) - ("crnn" in have) < 1:
            missing.append("a baseline (Step 3 Tesseract or Step 4 EMNIST)")
        print("Comparison table not made yet; missing results from:", " and ".join(missing))
        print("It is created automatically by whichever of Steps 2-4 finishes last.")
        return None
    systems = [(n, p) for n, p in SYSTEMS if p in have]
    per_split = {}
    for split in ("val", "test"):
        preds = {p: load_predictions(d / f"{p}_{split}_predictions.csv") for _, p in systems
                 if (d / f"{p}_{split}_predictions.csv").exists()}
        if "crnn" not in preds or len(preds) < 2:
            continue
        common = set.intersection(*({r["word_id"] for r in rows} for rows in preds.values()))
        rows = {p: sorted((r for r in v if r["word_id"] in common), key=lambda r: r["word_id"]) for p, v in preds.items()}
        per_split[split] = {p: _scores(v) for p, v in rows.items()}
        per_split[split]["_rows"] = rows
    params = {"crnn": rep.numbers().get("model", {}).get("parameters"),
              "emnist": have.get("emnist", {}).get("classifier", {}).get("parameters")}
    rows, nums = [], {}
    for name, p in systems:
        row = {"Model": name}
        for split in ("val", "test"):
            s = per_split.get(split, {}).get(p)
            row[f"{split.capitalize()} CER"] = pct(s["cer"]) if s else "--"
            row[f"{split.capitalize()} WER"] = pct(s["wer"]) if s else "--"
        ms = _speed(p, have[p])
        row["ms/word"] = "" if ms is None else (f"{ms:.2f}" if ms < 10 else f"{ms:.0f}")
        row["Params"] = f"{params[p] / 1e6:.2f}M" if params.get(p) else "--"
        rows.append(row)
        nums[name] = {s: per_split[s][p] for s in per_split if p in per_split[s]}
        nums[name]["ms_per_word"] = ms
    n_test = per_split["test"]["crnn"]["words"]
    full = have["crnn"].get("test", {}).get("words")
    note = None
    if full and n_test < full:
        note = (f"All models scored on the same {n_test:,} test words (the words every model was run on); "
                f"CRNN on all {full:,} test words: CER {pct(have['crnn']['test']['cer'])}%, "
                f"WER {pct(have['crnn']['test']['wer'])}%.")
    speed = {"crnn": f"CRNN batched (128) on {have['crnn'].get('device')}",
             "tesseract": "Tesseract per word on one CPU core",
             "emnist": f"EMNIST baseline: segmentation on the CPU + batched classification on "
                       f"{have.get('emnist', {}).get('device', 'GPU')}"}
    pre = {"tesseract": "Tesseract is pretrained on printed text", "emnist": "the EMNIST CNN is trained only on EMNIST characters"}
    base_txt = "; ".join(pre[p] for _, p in systems if p in pre)
    cap = (f"CRNN vs. the baselines on the same IAM words ({n_test:,} test words from writers unseen in training). "
           f"CER and WER in %, case-sensitive, lower is better. Speed: " + "; ".join(speed[p] for _, p in systems) +
           f". Neither baseline is trained on IAM ({base_txt}).")
    paths = tables.write_table(rows, list(rows[0]), d, "table_comparison", cap, "tab:comparison", note=note)
    rep.add(paths, "Main quantitative result: CRNN vs the baselines (" + ", ".join(n for n, p in systems if p != "crnn")
            + ") CER/WER on the same val/test words, speed and size.")
    sys_rows = per_split["test"]["_rows"]
    lens = {n: errors.cer_by_length([r["ref"] for r in sys_rows[p]], [r["hyp"] for r in sys_rows[p]])
            for n, p in systems if p in sys_rows}
    p2 = figures.plot_cer_by_length(lens, d, "fig_cer_by_length_comparison",
                                    title="CER by word length: " + " vs ".join(n.split(" (")[0] for n in lens)
                                    + " (same test words)")
    rep.add(p2, "CER by word length for " + ", ".join(lens) + " on the same test words.")
    nums["note"] = note
    nums["models"] = [n for n, _ in systems]
    nums["test_words"] = n_test
    rep.numbers("comparison", nums)
    return rows

### Command-line scripts (same functions, for use outside the notebook)

In [ ]:
%%writefile /content/aps360_htr/scripts/train.py
"""Train the CRNN on IAM words.

  python scripts/train.py --data-root /content/iam --out-dir runs/crnn
  python scripts/train.py --data-root data/fake_iam --epochs 2 --no-preload   # quick CPU check

Re-running with the same --out-dir resumes from last.pt (handy when Colab disconnects).
Pass --fresh to start over.
"""
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from src.config import ModelConfig, TrainConfig
from src.training.data import load_data
from src.training.engine import fit


def add_common_args(ap):
    ap.add_argument("--data-root", default=None)
    ap.add_argument("--split-dir", default="default", help="'none' = automatic writer-disjoint split")
    ap.add_argument("--epochs", type=int, default=TrainConfig.epochs)
    ap.add_argument("--batch-size", type=int, default=TrainConfig.batch_size)
    ap.add_argument("--lr", type=float, default=TrainConfig.lr)
    ap.add_argument("--hidden", type=int, default=ModelConfig.lstm_hidden)
    ap.add_argument("--dropout", type=float, default=ModelConfig.dropout)
    ap.add_argument("--patience", type=int, default=TrainConfig.patience)
    ap.add_argument("--num-workers", type=int, default=TrainConfig.num_workers)
    ap.add_argument("--train-frac", type=float, default=1.0)
    ap.add_argument("--max-batches", type=int, default=None)
    ap.add_argument("--no-preload", action="store_true")
    ap.add_argument("--no-check-images", action="store_true")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--seed", type=int, default=TrainConfig.seed)


def configs_from_args(a, out_dir):
    mc = ModelConfig(lstm_hidden=a.hidden, dropout=a.dropout)
    # Preloaded images are already in RAM, so worker processes would only add overhead.
    workers = a.num_workers if a.no_preload else 0
    tc = TrainConfig(epochs=a.epochs, batch_size=a.batch_size, lr=a.lr, patience=a.patience,
                     num_workers=workers,
                     train_frac=a.train_frac, max_batches=a.max_batches, amp=not a.no_amp,
                     preload=not a.no_preload, seed=a.seed, out_dir=Path(out_dir))
    return mc, tc


def main():
    ap = argparse.ArgumentParser()
    add_common_args(ap)
    ap.add_argument("--out-dir", default="outputs/runs/crnn")
    ap.add_argument("--fresh", action="store_true", help="ignore last.pt and start again")
    a = ap.parse_args()
    datasets, charset, _ = load_data(a.data_root, ("train", "val"), preload=not a.no_preload,
                                     check_images=not a.no_check_images, split_dir=a.split_dir)
    mc, tc = configs_from_args(a, a.out_dir)
    result = fit(datasets, charset, mc, tc, resume=not a.fresh)
    print(f"\nbest val CER {result['best_val_cer']:.4f}  ->  {result['out_dir']}/best.pt")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/aps360_htr/scripts/evaluate.py
"""Evaluate a checkpoint: CER/WER on a split, per-word speed, and every prediction.

  python scripts/evaluate.py --checkpoint runs/crnn/best.pt --data-root /content/iam --split test

Writes <out-dir>/<split>_metrics.json and <split>_predictions.csv (ref, hyp, edits),
and prints the most common mistakes, which is useful for the report's error analysis.
"""
import argparse
import csv
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from src.training.data import load_data
from src.training.engine import evaluate, get_device, load_model, make_loader
from src.utils.metrics import edit_distance


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--data-root", default=None)
    ap.add_argument("--split-dir", default="default")
    ap.add_argument("--split", default="test", choices=["train", "val", "test"])
    ap.add_argument("--out-dir", default=None, help="default: the checkpoint's folder")
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--no-check-images", action="store_true")
    a = ap.parse_args()

    device = get_device()
    model, charset, ck = load_model(a.checkpoint, device)
    print(f"checkpoint epoch {ck['epoch']} (val CER {ck['best_cer']:.4f})")
    datasets, _, _ = load_data(a.data_root, (a.split,), preload=False,
                               check_images=not a.no_check_images, split_dir=a.split_dir)
    loader, _ = make_loader(datasets[a.split], a.batch_size, shuffle=False)
    res = evaluate(model, loader, charset, device, amp=device.type == "cuda")

    out = Path(a.out_dir or Path(a.checkpoint).parent); out.mkdir(parents=True, exist_ok=True)
    metrics = {k: res[k] for k in ("cer", "wer", "loss", "n", "ms_per_word")}
    metrics.update(split=a.split, checkpoint=str(a.checkpoint), epoch=ck["epoch"])
    (out / f"{a.split}_metrics.json").write_text(json.dumps(metrics, indent=2))
    with open(out / f"{a.split}_predictions.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["reference", "prediction", "edits"])
        for r, h in zip(res["refs"], res["hyps"]):
            w.writerow([r, h, edit_distance(r, h)])

    print(f"\n{a.split}: {res['n']} words | CER {res['cer']:.4f} | WER {res['wer']:.4f} | "
          f"{res['ms_per_word']:.2f} ms/word (batched, {device})")
    wrong = [(r, h) for r, h in zip(res["refs"], res["hyps"]) if r != h]
    print(f"{len(wrong)} words wrong. Most common mistakes:")
    for (r, h), n in Counter(wrong).most_common(10):
        print(f"  {n:4d}x  {r!r} -> {h!r}")
    print(f"saved {out}/{a.split}_metrics.json and {a.split}_predictions.csv")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/aps360_htr/scripts/sweep.py
"""Hyperparameter sweep (same 12 settings as the Step 2 notebook), ranked by validation CER.

  python scripts/sweep.py --data-root /content/iam --out-dir runs/sweep --epochs 8 --train-frac 0.3

See src/training/sweep.py for the design. Results are appended to <out-dir>/sweep_results.csv as
each run finishes; re-running skips runs already there. The winner (lowest val CER) is printed
and is what the notebook's Train cell (choose_settings) uses. The test set is never touched.
"""
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from src.training.data import load_data
from src.training.sweep import choose_settings, run_sweep, sweep_configs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default=None)
    ap.add_argument("--split-dir", default="default")
    ap.add_argument("--out-dir", default="outputs/runs/sweep")
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--train-frac", type=float, default=0.3)
    ap.add_argument("--max-batches", type=int, default=None)
    ap.add_argument("--only", type=int, default=None, help="run only the first N settings (quick tests)")
    ap.add_argument("--hidden-scale", type=float, default=1.0, help=argparse.SUPPRESS)  # tests: smaller models
    ap.add_argument("--no-preload", action="store_true")
    ap.add_argument("--no-check-images", action="store_true")
    a = ap.parse_args()

    datasets, charset, _ = load_data(a.data_root, ("train", "val"), preload=not a.no_preload,
                                     check_images=not a.no_check_images, split_dir=a.split_dir)
    out = Path(a.out_dir); out.mkdir(parents=True, exist_ok=True)
    configs = sweep_configs()[: a.only] if a.only else sweep_configs()
    if a.hidden_scale != 1.0:
        configs = [{**c, "lstm_hidden": max(8, int(c["lstm_hidden"] * a.hidden_scale))} for c in configs]
    csv_path = out / "sweep_results.csv"
    ranked = run_sweep(datasets, charset, csv_path, out, configs, epochs=a.epochs, train_frac=a.train_frac,
                       max_batches=a.max_batches)
    print("\nRanked by val CER:")
    for i, r in enumerate(ranked, 1):
        print(f"  {i:2d}. {r['name']:28s} CER {r['best_val_cer']:.4f}  WER {r['best_val_wer']:.4f}  ({r['minutes']:.1f} min)")
    settings, info = choose_settings(csv_path)
    print(f"\nTrain would use ({info['source']}): {settings}\n  {info['reason']}")
    print(f"saved {csv_path}")


if __name__ == "__main__":
    main()

## Install missing packages
Colab already has PyTorch, OpenCV, NumPy, Pillow, matplotlib, pyarrow and pytest, so this only installs what's missing.
`editdistance` (faster CER/WER) is optional; the code falls back to pure Python without it.

In [ ]:
import importlib.util, subprocess, sys
need = {"torch": "torch", "torchvision": "torchvision", "cv2": "opencv-python", "numpy": "numpy",
        "PIL": "pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "pytest": "pytest",
        "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
        "pytesseract": "pytesseract", "editdistance": "editdistance"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
print("missing:", missing or "nothing")
for pkg in missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    note = " (optional: CER/WER use the pure-Python fallback)" if pkg == "editdistance" else ""
    print(f"  {pkg}: {'installed' if r.returncode == 0 else 'install failed' + note}")

## Unpack archive.zip (about 1.2 GB)
Copied to Colab's disk and unzipped there, which takes a few minutes. Re-running in the same session skips it.

In [ ]:
import os, shutil, time
marker = os.path.join(RAW_DIR, ".unzipped")
if os.path.exists(marker):
    print("Already unzipped this session:", RAW_DIR)
else:
    t = time.time()
    shutil.copy(ARCHIVE, "/content/archive.zip")
    print(f"copied in {time.time()-t:.0f}s"); t = time.time()
    !unzip -q -o /content/archive.zip -d {RAW_DIR}
    os.remove("/content/archive.zip")
    open(marker, "w").close()
    print(f"unzipped in {time.time()-t:.0f}s")
!ls {RAW_DIR}

## Find words.txt and the images
`setup_iam.py` finds the word images and a `words.txt` in any download layout (official tgz, Kaggle `iam_words/`, or a flat folder) and links them into `/content/iam`.

**Which labels get used** (first one that exists wins):
1. `words.txt` in your project folder: the full official file, if you download it from IAM;
2. `words_recovered.txt` in your project folder, made by the label-recovery step below;
3. the `words.txt` inside `archive.zip`. The Kaggle copy is **cut off at e07-072**, so only ~39% of IAM has labels.

In [ ]:
def pick_words_txt():
    for p in (OFFICIAL_WORDS_TXT, RECOVERED_WORDS_TXT):
        if os.path.exists(p):
            return p
    return None

def run_setup():
    wt = pick_words_txt()
    print("labels from:", wt or "the words.txt inside archive.zip")
    extra = f'--words-txt "{wt}"' if wt else ""
    !python scripts/setup_iam.py --raw {RAW_DIR} --out {DATA_ROOT} {extra}

run_setup()

## Recover the missing labels (only needed once)
**Why:** the Kaggle `words.txt` stops part-way through line `e07-072-09-`, but `archive.zip` has the images for all of IAM. Without the missing lines, about 60% of the images can't be used.

**How:** every original IAM PNG has its form's XML stored *inside the file* (a PNG text chunk). That XML gives the word's ID, its text, its tag, its bounding box and its line's segmentation status (ok/err), which are exactly the fields of a `words.txt` line. `recover_labels.py`:
1. first tries the XML inside **your own** images (fast, no download);
2. if they've been stripped, it downloads the Hugging Face copy of IAM ([priyank-m/IAM_words_text_recognition](https://huggingface.co/datasets/priyank-m/IAM_words_text_recognition), ~1.2 GB), whose PNGs still carry the XML, and matches **by word ID**. Images whose ID can't be found fall back to a pixel-hash match.

**Checks it prints, for the report:**
- *rebuilt vs original*: for words that have both, the rebuilt line must equal the original Kaggle line (expect ~100%);
- *pixel check*: the HF image is pixel-identical to yours for a sample of recovered words;
- matched / unmatched / ambiguous counts. Below 90%, nothing is written and training uses the labelled subset.

The result is saved to your project folder as `words_recovered.txt` (original lines first, then a `#--- recovered entries below ---` section), so later sessions and the Step 2 notebook reuse it. Recovered words keep their real IAM IDs, so the **Aachen writer-disjoint splits still apply**.

In [ ]:
if os.path.exists(OFFICIAL_WORDS_TXT):
    print("Using the official full words.txt; no recovery needed.")
elif os.path.exists(RECOVERED_WORDS_TXT):
    print("Reusing", RECOVERED_WORDS_TXT)
else:
    !python scripts/recover_labels.py --data-root {DATA_ROOT} --out "{RECOVERED_WORDS_TXT}" \
        --hf-cache /content/hf_iam --report "{OUTPUT_DIR}/label_recovery.json" \
        --preview "{OUTPUT_DIR}/label_recovery_preview.png"
    if os.path.exists(RECOVERED_WORDS_TXT):
        run_setup()          # re-link with the recovered labels
        from IPython.display import Image as _I, display
        display(_I(os.path.join(OUTPUT_DIR, "label_recovery_preview.png")))

## Run the tests
All tests, including the model (output shapes, CTC lengths, learning a toy batch) and a 2-epoch training, resume and evaluate run on fake data. Every line should say `passed`.

In [ ]:
!python -m pytest -q tests

## Load the data
`load_data()` is Step 1's `make_datasets()` (same filters, Aachen splits, writer table), then it **preloads** the train and val images into RAM, resized to 32 px (a few hundred MB for all of IAM, ~2–4 minutes). Colab has only 2 CPU cores, so decoding PNGs every epoch would leave the GPU waiting.

**Broken image files.** A few IAM word PNGs are empty or corrupt (e.g. `a01-117-05-02`, `r06-022-03-05`). They can't crash this cell any more:
- known broken IDs are always dropped (a hardcoded list in `src/data/iam.py`);
- `check_images=False` still checks that every file exists and isn't empty (a few seconds instead of opening all ~115k images);
- any other file that fails to decode during preloading is dropped and listed by name.

All of them are counted as `bad_or_missing_image` in the data-filtering table.

In [ ]:
from src.training.data import load_data
datasets, charset, stats = load_data(DATA_ROOT, splits=("train", "val"), preload=True, check_images=False)
print({k: stats[k] for k in ("kept", "segmentation_err", "bad_or_missing_image", "malformed_line", "writer_source", "split_method")})
print("bad images dropped:", stats.get("bad_image_ids", []))

## Sanity check: memorise one batch
Before a long run, check the model can drive the loss towards 0 on 16 fixed words (no dropout). If it can't, something is wired wrong (labels, CTC lengths, decoding), and training would just waste GPU time. Takes ~30 s on a T4.

In [ ]:
import matplotlib.pyplot as plt
from src.config import ModelConfig
from src.training.engine import overfit_one_batch
losses, pairs = overfit_one_batch(datasets["train"], charset, ModelConfig(dropout=0.0), steps=300)
plt.figure(figsize=(5, 3)); plt.plot(losses); plt.xlabel("step"); plt.ylabel("CTC loss"); plt.title("One batch"); plt.show()
print(f"loss {losses[0]:.2f} -> {losses[-1]:.3f}")
print("memorised:", sum(r == h for r, h in pairs), "/", len(pairs), pairs[:6])

## Hyperparameter sweep
Short runs that compare settings, ranked by **validation** CER only (the test set is never used to choose). The **winner feeds straight into the Train cell below**.

**Design (12 runs, not a full grid):**
- a small grid: learning rate {5e-4, 1e-3, 3e-3} × LSTM hidden size {128, 256} (6 runs);
- one-at-a-time changes around the default (lr 1e-3, hidden 256, dropout 0.3, batch 64, 2 LSTM layers): dropout 0.1 and 0.5, batch 32 and 128, hidden 384, 3 LSTM layers (6 runs).

Every run trains for `SWEEP_EPOCHS = 8` epochs on the same 30% of the training words and is scored on the full validation set. Short runs rank settings by how fast and how well they learn, which is what a sweep can afford; the final model then trains to convergence.

**Time (estimate, not measured on a T4):** about 1.5–3 min per run (batch 32, hidden 384 and 3 layers are the slowest), so roughly **25–45 min** for all 12. After each run the cell prints the time taken and an estimate for the rest.

**Resumable:** each finished run is saved to `outputs/sweep_results.csv` on Drive immediately, and runs already there are skipped. After a disconnect, Run all again and only the unfinished runs train (the interrupted one resumes from its `last.pt`). If all 12 are in the CSV, nothing re-runs. Set `RUN_SWEEP = False` to skip it: then the Train cell still uses an existing `sweep_results.csv`, or the defaults if there is none.

In [ ]:
from src.training.sweep import run_sweep, sweep_configs, run_name
RUN_SWEEP = True
SWEEP_EPOCHS = 8
SWEEP_TRAIN_FRAC = 0.3
SWEEP_CSV = os.path.join(OUTPUT_DIR, "sweep_results.csv")
configs = sweep_configs()
print(f"{len(configs)} sweep settings:")
for c in configs:
    print(f"  {run_name(c):28s} ({c['group']})")
if RUN_SWEEP:
    ranked = run_sweep(datasets, charset, SWEEP_CSV, os.path.join(PROJECT_DIR, "runs", "sweep"), configs,
                       epochs=SWEEP_EPOCHS, train_frac=SWEEP_TRAIN_FRAC)
    print("\nranked by validation CER (best first):")
    for i, r in enumerate(ranked, 1):
        print(f"  {i:2d}. {r['name']:28s} val CER {r['best_val_cer']:.4f}  WER {r['best_val_wer']:.4f}  "
              f"best epoch {r['best_epoch']}  {r['minutes']:.1f} min")
else:
    print("RUN_SWEEP is False: not training any sweep runs.")

## Train
**Settings:** if `sweep_results.csv` exists (from this session or an earlier one), this cell uses the **best row** (lowest val CER, ties broken by val WER). Otherwise, or if no sweep run learned anything (val CER ≥ 99%), it uses the defaults: lr 1e-3, hidden 256, dropout 0.3, batch 64, 2 LSTM layers. It prints which settings it chose and why.

- **Optimizer:** AdamW (weight decay 1e-4). The LR halves after 3 epochs without a val-CER improvement; training stops after 8.
- **Loss:** CTC (blank = 0) with `zero_infinity=True`, so a label that can't fit its image can't produce an infinite loss.
- **Speed:** mixed precision (float16) on the GPU, batches of similar-width words, images preloaded in RAM.
- **Checkpoints** go to `RUN_DIR` on Drive: `best.pt` (lowest val CER), `last.pt` (every epoch), `history.csv`, `config.json`.
- **`RUN_NAME` includes the settings** (e.g. `crnn_lr0.001_h256_d0.3_b64_L2`), so different settings never share a folder. Before training, `check_run_dir()` also compares `config.json` with the current settings and stops with a clear message instead of silently resuming a checkpoint trained differently.
- **If Colab disconnects:** Run all again. The sweep is skipped (already in the CSV), the same winner is picked, and this cell **resumes** from `last.pt`.

**Time (estimate, not yet measured on a T4):** roughly 0.5–1 min per epoch on all of IAM (~48k training words), so ~20–45 min before early stopping (usually 25–40 epochs). The cut-off subset (~31k words) is about a third faster.

In [ ]:
import json
from src.training.engine import fit
from src.training.sweep import choose_settings, make_configs, check_run_dir
EPOCHS = 40
SETTINGS, SETTINGS_INFO = choose_settings(SWEEP_CSV)
print(f"Settings from: {SETTINGS_INFO['source'].upper()} - {SETTINGS_INFO['reason']}")
for k, v in SETTINGS.items():
    print(f"  {k:12s} = {v:g}" if isinstance(v, float) else f"  {k:12s} = {v}")

RUN_NAME = "crnn_" + run_name(SETTINGS)          # the settings are part of the folder name
RUN_DIR  = os.path.join(PROJECT_DIR, "runs", RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)
model_cfg, train_cfg = make_configs(SETTINGS, epochs=EPOCHS, patience=8, num_workers=0, out_dir=RUN_DIR)
print("RUN_DIR:", RUN_DIR, "|", check_run_dir(RUN_DIR, model_cfg, train_cfg))
json.dump({"settings": SETTINGS, **{k: v for k, v in SETTINGS_INFO.items()}},
          open(os.path.join(RUN_DIR, "settings_choice.json"), "w"), indent=2, default=str)
result = fit(datasets, charset, model_cfg, train_cfg, resume=True)
print("best val CER:", result["best_val_cer"])

## Training curves
Loss should fall for both train and val. If val loss rises while train loss keeps falling, the model is overfitting; early stopping keeps the best epoch. (Report-quality versions of these curves are saved in the **Report outputs** section below.)

In [ ]:
from IPython.display import Image, display
from src.utils.plots import plot_history
png = plot_history(os.path.join(RUN_DIR, "history.csv"), os.path.join(RUN_DIR, "curves.png"))
display(Image(png))

## Evaluate on validation and test
`best.pt` is reloaded and greedy-decoded on **val** and then on **test** (unseen writers, Aachen split). Test is used only here, once training is finished.
- **CER / WER** are corpus-level: total edits divided by total reference characters (or words).
- **ms/word** is batched GPU throughput; the single-word latency below is closer to the live webcam case (brief target: under ~1 s per word).
- Every prediction is also saved **with its word ID** (`crnn_val_predictions.csv`, `crnn_test_predictions.csv` in `outputs/report/`), so Step 3 can score Tesseract on exactly the same words.

In [ ]:
import csv, json, time, torch
from src.training.engine import evaluate, get_device, load_model, make_loader
from src.training.data import load_data
from src.report.predict import predict_dataset, save_predictions
device = get_device()
DEVICE_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
model, cs, ck = load_model(os.path.join(RUN_DIR, "best.pt"), device)
print(f"best epoch {ck['epoch']}, val CER {ck['best_cer']:.4f}")
from src.training.data import add_dropped
test_sets, _, _ = load_data(DATA_ROOT, splits=("test",), preload=True, check_images=False)
datasets["test"] = test_sets["test"]      # unreadable test images (if any) are dropped here...
add_dropped(stats, test_sets)             # ...and counted as bad images in the data-filtering table
eval_sets = {"val": datasets["val"], "test": test_sets["test"]}
results, preds = {}, {}
os.makedirs(REPORT_DIR, exist_ok=True)
for split, ds in eval_sets.items():
    loader, _ = make_loader(ds, 128, shuffle=False)
    r = evaluate(model, loader, cs, device, amp=device.type == "cuda")
    results[split] = r
    preds[split] = predict_dataset(model, ds, cs, device, batch_size=128, amp=device.type == "cuda")
    save_predictions(preds[split], os.path.join(REPORT_DIR, f"crnn_{split}_predictions.csv"))
    print(f"{split:4s}: {r['n']} words | CER {r['cer']:.4f} | WER {r['wer']:.4f} | {r['ms_per_word']:.2f} ms/word (batched)")

# single-word latency (one image at a time, like the live demo)
x = test_sets["test"][0][0].unsqueeze(0).to(device)
with torch.no_grad():
    for _ in range(5): model(x)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(50): model(x)
    if device.type == "cuda": torch.cuda.synchronize()
latency_ms = (time.time() - t0) / 50 * 1000
print(f"single-word latency: {latency_ms:.1f} ms on {DEVICE_NAME}")

metrics = {s: {k: r[k] for k in ("cer", "wer", "n", "ms_per_word")} for s, r in results.items()}
metrics["single_word_latency_ms"] = latency_ms
metrics["device"] = DEVICE_NAME
json.dump(metrics, open(os.path.join(RUN_DIR, "metrics.json"), "w"), indent=2)
t = results["test"]
with open(os.path.join(RUN_DIR, "test_predictions.csv"), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["reference", "prediction"]); w.writerows(zip(t["refs"], t["hyps"]))
print("saved metrics.json and test_predictions.csv to", RUN_DIR)

# Report outputs
Everything the progress report needs is saved to **`outputs/report/`** on Drive (separate from the checkpoints in `runs/`):
figures as **PNG (300 dpi) + PDF**, tables as **CSV + LaTeX booktabs `.tex`** (complete `table` environments you can `\input{}`),
`report_numbers.json` (every key number in one place) and `report_summary.md` (every file with a suggested caption).

Everything is rebuilt from what is on Drive (`history.csv`, `best.pt`) plus the evaluation above, so it is the same whether training ran in one go or was resumed. Re-running these cells just overwrites the files.

Figures are drawn for the ICLR template's 5.5-inch text width with 8-9 pt fonts: use `\includegraphics[width=\linewidth]{fig_....pdf}` (or `0.48\linewidth` for the half-width ones, side by side).

In [ ]:
from IPython.display import Image, Markdown, display
from src.report.manifest import Report
from src.report import crnn_report as cr, compare
from src.report.figures import read_history
rep = Report(REPORT_DIR, step=2)
def show(paths):
    for p in paths or []:
        if p.endswith(".png"):
            display(Image(p, width=700))
from src.report.tables import csv_to_markdown
show_table = lambda paths: display(Markdown(csv_to_markdown([p for p in paths if p.endswith(".csv")][0])))
print("report folder:", REPORT_DIR)

### Training curves (from `history.csv`)
`fig_training_curves` (both panels), `fig_loss_curve` and `fig_val_cer_wer_curve` (half-width). The dotted line is the best epoch, whose weights are in `best.pt`.

In [ ]:
curve_paths, history = cr.save_curves(rep, os.path.join(RUN_DIR, "history.csv"))
show(curve_paths[:1])

### Final metrics, model summary and hyperparameters
`table_crnn_metrics` (quantitative results), `table_model_summary` (layer-by-layer output shapes and parameters: the rubric's *model complexity*) and `table_hyperparameters`.

In [ ]:
nums = cr.save_metrics(rep, preds, results, latency_ms, DEVICE_NAME)
layers, total, macs = cr.save_model_summary(rep, model, cs, ck, train_cfg, model_cfg, history, DEVICE_NAME,
                                            settings_info=SETTINGS_INFO)
print(f"{total:,} parameters ({total * 4 / 1e6:.1f} MB as float32), {macs / 1e6:.0f} M multiply-accumulates per 128-px-wide word")
for name in ("table_crnn_metrics", "table_model_summary", "table_hyperparameters"):
    print(name); show_table([os.path.join(REPORT_DIR, name + ".csv")])

### Data statistics and a cleaned sample
`table_data_filtering` (what each filter removed), `table_data_splits` (words, writers and forms per split), `fig_label_length_hist` (train label lengths) and `fig_preprocessing_examples` (the rubric's *example of a cleaned training sample*: original crop vs. the 32-px CRNN input).

In [ ]:
label_source = os.path.basename(pick_words_txt() or "words.txt inside archive.zip")
cr.save_data_stats(rep, stats, datasets, label_source)
show_table([os.path.join(REPORT_DIR, "table_data_filtering.csv")]); show_table([os.path.join(REPORT_DIR, "table_data_splits.csv")])
show(cr.save_label_hist(rep, datasets))
show(cr.save_preprocessing_examples(rep, datasets["train"]))

### Qualitative results
`fig_crnn_predictions`: 12 **randomly chosen** test words (not cherry-picked) at their original resolution, with the true text and the prediction (green = right, red = wrong).

In [ ]:
show(cr.save_prediction_grid(rep, test_sets["test"], preds["test"], n=12, seed=0))

### Error analysis
- `fig_crnn_char_confusions`: the most frequent character **substitutions** (e.g. `a → o`), with the total substitutions / deletions / insertions in the title;
- `table_crnn_char_confusions` (all error types, ∅ = missing/extra character) and `table_crnn_word_errors` (most common wrong words);
- `fig_crnn_cer_by_length` + `table_crnn_cer_by_length`: CER per word length (short words and punctuation are often the hardest).

In [ ]:
conf, breakdown, err_paths = cr.save_error_analysis(rep, preds["test"])
print("test errors:", breakdown)
show(err_paths[:2]); show_table([os.path.join(REPORT_DIR, "table_crnn_word_errors.csv")])
show(cr.save_cer_by_length(rep, preds["test"])[:2])

### Architecture diagram
`fig_crnn_architecture`: block diagram with the tensor shape after every stage, drawn from the model settings (so it stays correct if you change channels or hidden size).

In [ ]:
show(cr.save_architecture(rep, model_cfg, cs.num_classes))

### Sweep table and CRNN vs baselines comparison
`table_sweep` (every swept setting, best first, the chosen row in bold with a *) and `fig_sweep` (val CER per run, chosen run in orange), if a sweep exists. `table_comparison` + `fig_cer_by_length_comparison` include every baseline that has results (Step 3 Tesseract, Step 4 EMNIST); whichever notebook runs last makes them.

In [ ]:
chosen = SETTINGS_INFO["winner"]["name"] if SETTINGS_INFO["source"] == "sweep" else None
show(cr.save_sweep(rep, SWEEP_CSV, chosen_name=chosen))
if os.path.exists(os.path.join(REPORT_DIR, "table_sweep.csv")):
    show_table([os.path.join(REPORT_DIR, "table_sweep.csv")])
if compare.save_comparison(rep):
    show_table([os.path.join(REPORT_DIR, "table_comparison.csv")])

### Summary and list of files
`report_summary.md` lists every file with a one-line suggested caption; `report_numbers.json` holds every number above.

In [ ]:
from src.report.manifest import future_work
summary = rep.write_summary(future_work(rep))   # the EMNIST item drops out once Step 4 has results
display(Markdown(open(summary).read()))

In [ ]:
print("Saved report files:")
rep.list_files();

## Next
- Run **APS360 - HTR Step 3 (Tesseract Baseline)** (CPU runtime is fine) and **Step 4 (EMNIST baseline)** (GPU). Each adds its baseline files to the same `outputs/report/` folder, and the last one to finish rebuilds the **CRNN vs baselines** table with all three models.
- Then upload `outputs/report/` to Overleaf (see `report_summary.md`).
- **Later (augmentation):** webcam-style distortions for training only, then the with/without ablation (`src/data/transforms.py` TODO).